In [1]:
# ============================================================
# STEP 1: LOADING MODELING-READY DATA
# ============================================================

import pandas as pd
import numpy as np

print("=" * 60)
print("LOADING MODELING-READY DATA")
print("=" * 60)


# ------------------------------------------------------------
# Load selected features
# ------------------------------------------------------------

X_train = pd.read_csv(
    "data/processed/X_train_selected.csv"
)

X_test = pd.read_csv(
    "data/processed/X_test_selected.csv"
)


# ------------------------------------------------------------
# Load target
# ------------------------------------------------------------

y_train = pd.read_csv(
    "data/processed/y_train.csv"
).squeeze()

y_test = pd.read_csv(
    "data/processed/y_test.csv"
).squeeze()


# ------------------------------------------------------------
# Display shapes
# ------------------------------------------------------------

print("\n📌 TRAINING DATA")
print("-" * 60)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")


print("\n📌 TESTING DATA")
print("-" * 60)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


# ------------------------------------------------------------
# Display features
# ------------------------------------------------------------

print("\n📌 MODEL FEATURES")
print("-" * 60)

for i, feature in enumerate(
    X_train.columns,
    start=1
):
    print(f"{i:2d}. {feature}")


# ------------------------------------------------------------
# Target distribution
# ------------------------------------------------------------

print("\n📌 TRAINING TARGET DISTRIBUTION")
print("-" * 60)

print(
    y_train.value_counts()
)


print("\n📌 TRAINING TARGET PROPORTION")
print("-" * 60)

print(
    (y_train.value_counts(normalize=True) * 100)
    .round(2)
)


print("\n📌 TESTING TARGET DISTRIBUTION")
print("-" * 60)

print(
    y_test.value_counts()
)


print("\n📌 DATA TYPES")
print("-" * 60)

print(
    X_train.dtypes
)


print("\n📌 MISSING VALUES")
print("-" * 60)

print(
    f"X_train missing values: "
    f"{X_train.isnull().sum().sum()}"
)

print(
    f"X_test missing values: "
    f"{X_test.isnull().sum().sum()}"
)


print("\n✅ MODELING DATA LOADED SUCCESSFULLY!")

LOADING MODELING-READY DATA

📌 TRAINING DATA
------------------------------------------------------------
X_train shape: (45000, 11)
y_train shape: (45000,)

📌 TESTING DATA
------------------------------------------------------------
X_test shape: (5000, 11)
y_test shape: (5000,)

📌 MODEL FEATURES
------------------------------------------------------------
 1. numerical__Communication_Skills
 2. numerical__CGPA
 3. numerical__Backlogs
 4. numerical__Career_Development_Index
 5. numerical__Coding_Skills
 6. numerical__Projects
 7. numerical__Certifications
 8. numerical__Academic_Strength
 9. numerical__Aptitude_Test_Score
10. numerical__Professional_Skill_Index
11. numerical__Practical_Experience_Index

📌 TRAINING TARGET DISTRIBUTION
------------------------------------------------------------
Placement_Status
Not Placed    28688
Placed        16312
Name: count, dtype: int64

📌 TRAINING TARGET PROPORTION
------------------------------------------------------------
Placement_Status
Not

In [2]:
# ============================================================
# ESTABLISHING PLACEMENT READINESS BASELINE
# ============================================================

print("=" * 60)
print("ESTABLISHING PLACEMENT READINESS BASELINE")
print("=" * 60)

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ------------------------------------------------------------
# BASELINE MODEL
# ------------------------------------------------------------

baseline_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

# ------------------------------------------------------------
# BASELINE PERFORMANCE
# ------------------------------------------------------------

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

baseline_precision = precision_score(
    y_test,
    baseline_predictions,
    pos_label="Placed",
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_predictions,
    pos_label="Placed",
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_predictions,
    pos_label="Placed",
    zero_division=0
)

# ROC-AUC requires probability scores
baseline_probabilities = baseline_model.predict_proba(X_test)[:, 1]

baseline_roc_auc = roc_auc_score(
    (y_test == "Placed").astype(int),
    baseline_probabilities
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("\n📌 BASELINE MODEL")
print("-" * 60)
print("Strategy: Most Frequent Class")

print("\n📌 BASELINE PERFORMANCE")
print("-" * 60)

print(f"Accuracy  : {baseline_accuracy:.4f}")
print(f"Precision : {baseline_precision:.4f}")
print(f"Recall    : {baseline_recall:.4f}")
print(f"F1-Score  : {baseline_f1:.4f}")
print(f"ROC-AUC   : {baseline_roc_auc:.4f}")

print("\n📌 BASELINE CLASSES")
print("-" * 60)
print(baseline_model.classes_)

print("\n📌 BASELINE PREDICTION DISTRIBUTION")
print("-" * 60)
print(
    pd.Series(
        baseline_predictions
    ).value_counts()
)

print("\n" + "=" * 60)
print("✅ PLACEMENT READINESS BASELINE CREATED!")
print("=" * 60)

ESTABLISHING PLACEMENT READINESS BASELINE

📌 BASELINE MODEL
------------------------------------------------------------
Strategy: Most Frequent Class

📌 BASELINE PERFORMANCE
------------------------------------------------------------
Accuracy  : 0.6376
Precision : 0.0000
Recall    : 0.0000
F1-Score  : 0.0000
ROC-AUC   : 0.5000

📌 BASELINE CLASSES
------------------------------------------------------------
['Not Placed' 'Placed']

📌 BASELINE PREDICTION DISTRIBUTION
------------------------------------------------------------
Not Placed    5000
Name: count, dtype: int64

✅ PLACEMENT READINESS BASELINE CREATED!


In [3]:
# ============================================================
# STEP 2: PLACEMENT READINESS BASELINE
# ============================================================

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 60)
print("ESTABLISHING PLACEMENT READINESS BASELINE")
print("=" * 60)


# ------------------------------------------------------------
# Create baseline model
# ------------------------------------------------------------

baseline_model = DummyClassifier(
    strategy="most_frequent"
)


# ------------------------------------------------------------
# Train baseline
# ------------------------------------------------------------

baseline_model.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

baseline_predictions = baseline_model.predict(
    X_test
)

baseline_probabilities = baseline_model.predict_proba(
    X_test
)


# ------------------------------------------------------------
# Identify the probability column for "Placed"
# ------------------------------------------------------------

placed_class_index = list(
    baseline_model.classes_
).index("Placed")

placed_probabilities = (
    baseline_probabilities[:, placed_class_index]
)


# ------------------------------------------------------------
# Calculate evaluation metrics
# ------------------------------------------------------------

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

baseline_precision = precision_score(
    y_test,
    baseline_predictions,
    pos_label="Placed",
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_predictions,
    pos_label="Placed",
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_predictions,
    pos_label="Placed",
    zero_division=0
)

baseline_roc_auc = roc_auc_score(
    y_test,
    placed_probabilities
)


# ------------------------------------------------------------
# Display baseline results
# ------------------------------------------------------------

print("\n📌 BASELINE MODEL")
print("-" * 60)

print("Strategy: Most Frequent Class")


print("\n📌 BASELINE PERFORMANCE")
print("-" * 60)

print(f"Accuracy  : {baseline_accuracy:.4f}")
print(f"Precision : {baseline_precision:.4f}")
print(f"Recall    : {baseline_recall:.4f}")
print(f"F1-Score  : {baseline_f1:.4f}")
print(f"ROC-AUC   : {baseline_roc_auc:.4f}")


print("\n📌 BASELINE CLASSES")
print("-" * 60)

print(
    baseline_model.classes_
)


print("\n📌 BASELINE PREDICTION DISTRIBUTION")
print("-" * 60)

print(
    pd.Series(
        baseline_predictions
    ).value_counts()
)


print("\n✅ PLACEMENT READINESS BASELINE CREATED!")

ESTABLISHING PLACEMENT READINESS BASELINE

📌 BASELINE MODEL
------------------------------------------------------------
Strategy: Most Frequent Class

📌 BASELINE PERFORMANCE
------------------------------------------------------------
Accuracy  : 0.6376
Precision : 0.0000
Recall    : 0.0000
F1-Score  : 0.0000
ROC-AUC   : 0.5000

📌 BASELINE CLASSES
------------------------------------------------------------
['Not Placed' 'Placed']

📌 BASELINE PREDICTION DISTRIBUTION
------------------------------------------------------------
Not Placed    5000
Name: count, dtype: int64

✅ PLACEMENT READINESS BASELINE CREATED!


In [4]:
# ============================================================
# STEP 3: LOGISTIC REGRESSION - PLACEMENT READINESS MODEL
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("TRAINING LOGISTIC REGRESSION MODEL")
print("=" * 60)


# ------------------------------------------------------------
# Create Logistic Regression model
# ------------------------------------------------------------

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# ------------------------------------------------------------
# Train model
# ------------------------------------------------------------

logistic_model.fit(
    X_train,
    y_train
)

print("\n✅ Logistic Regression model trained successfully!")


# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

logistic_predictions = logistic_model.predict(
    X_test
)

logistic_probabilities = logistic_model.predict_proba(
    X_test
)


# ------------------------------------------------------------
# Get probability of "Placed"
# ------------------------------------------------------------

placed_class_index = list(
    logistic_model.classes_
).index("Placed")

logistic_placed_probability = (
    logistic_probabilities[:, placed_class_index]
)


# ------------------------------------------------------------
# Calculate performance metrics
# ------------------------------------------------------------

logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

logistic_precision = precision_score(
    y_test,
    logistic_predictions,
    pos_label="Placed",
    zero_division=0
)

logistic_recall = recall_score(
    y_test,
    logistic_predictions,
    pos_label="Placed",
    zero_division=0
)

logistic_f1 = f1_score(
    y_test,
    logistic_predictions,
    pos_label="Placed",
    zero_division=0
)

logistic_roc_auc = roc_auc_score(
    y_test,
    logistic_placed_probability
)


# ------------------------------------------------------------
# Display performance
# ------------------------------------------------------------

print("\n📌 LOGISTIC REGRESSION PERFORMANCE")
print("-" * 60)

print(f"Accuracy  : {logistic_accuracy:.4f}")
print(f"Precision : {logistic_precision:.4f}")
print(f"Recall    : {logistic_recall:.4f}")
print(f"F1-Score  : {logistic_f1:.4f}")
print(f"ROC-AUC   : {logistic_roc_auc:.4f}")


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\n📌 CLASSIFICATION REPORT")
print("-" * 60)

print(
    classification_report(
        y_test,
        logistic_predictions,
        target_names=["Not Placed", "Placed"],
        zero_division=0
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

print("\n📌 CONFUSION MATRIX")
print("-" * 60)

cm_logistic = confusion_matrix(
    y_test,
    logistic_predictions,
    labels=["Not Placed", "Placed"]
)

print(cm_logistic)


# ------------------------------------------------------------
# Prediction distribution
# ------------------------------------------------------------

print("\n📌 PREDICTION DISTRIBUTION")
print("-" * 60)

print(
    pd.Series(
        logistic_predictions
    ).value_counts()
)


print("\n✅ LOGISTIC REGRESSION ANALYSIS COMPLETED!")

TRAINING LOGISTIC REGRESSION MODEL

✅ Logistic Regression model trained successfully!

📌 LOGISTIC REGRESSION PERFORMANCE
------------------------------------------------------------
Accuracy  : 0.8696
Precision : 0.8212
Recall    : 0.8184
F1-Score  : 0.8198
ROC-AUC   : 0.9360

📌 CLASSIFICATION REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

  Not Placed       0.90      0.90      0.90      3188
      Placed       0.82      0.82      0.82      1812

    accuracy                           0.87      5000
   macro avg       0.86      0.86      0.86      5000
weighted avg       0.87      0.87      0.87      5000


📌 CONFUSION MATRIX
------------------------------------------------------------
[[2865  323]
 [ 329 1483]]

📌 PREDICTION DISTRIBUTION
------------------------------------------------------------
Not Placed    3194
Placed        1806
Name: count, dtype: int64

✅ LOGISTIC REGRESSION ANALYSIS COMPLETED!


In [5]:
# ============================================================
# STEP 4: DECISION TREE - PLACEMENT READINESS MODEL
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("TRAINING DECISION TREE MODEL")
print("=" * 60)


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

decision_tree_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=6,
    min_samples_split=20,
    min_samples_leaf=10
)


# ------------------------------------------------------------
# Train model
# ------------------------------------------------------------

decision_tree_model.fit(
    X_train,
    y_train
)

print("\n✅ Decision Tree model trained successfully!")


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

tree_predictions = decision_tree_model.predict(
    X_test
)

tree_probabilities = decision_tree_model.predict_proba(
    X_test
)


# ------------------------------------------------------------
# Probability of "Placed"
# ------------------------------------------------------------

placed_class_index = list(
    decision_tree_model.classes_
).index("Placed")

tree_placed_probability = (
    tree_probabilities[:, placed_class_index]
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

tree_accuracy = accuracy_score(
    y_test,
    tree_predictions
)

tree_precision = precision_score(
    y_test,
    tree_predictions,
    pos_label="Placed",
    zero_division=0
)

tree_recall = recall_score(
    y_test,
    tree_predictions,
    pos_label="Placed",
    zero_division=0
)

tree_f1 = f1_score(
    y_test,
    tree_predictions,
    pos_label="Placed",
    zero_division=0
)

tree_roc_auc = roc_auc_score(
    y_test,
    tree_placed_probability
)


# ------------------------------------------------------------
# Display performance
# ------------------------------------------------------------

print("\n📌 DECISION TREE PERFORMANCE")
print("-" * 60)

print(f"Accuracy  : {tree_accuracy:.4f}")
print(f"Precision : {tree_precision:.4f}")
print(f"Recall    : {tree_recall:.4f}")
print(f"F1-Score  : {tree_f1:.4f}")
print(f"ROC-AUC   : {tree_roc_auc:.4f}")


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\n📌 CLASSIFICATION REPORT")
print("-" * 60)

print(
    classification_report(
        y_test,
        tree_predictions,
        target_names=["Not Placed", "Placed"],
        zero_division=0
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

print("\n📌 CONFUSION MATRIX")
print("-" * 60)

tree_cm = confusion_matrix(
    y_test,
    tree_predictions,
    labels=["Not Placed", "Placed"]
)

print(tree_cm)


# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

tree_feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": decision_tree_model.feature_importances_
})

tree_feature_importance = (
    tree_feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n📌 DECISION TREE FEATURE IMPORTANCE")
print("-" * 60)

print(
    tree_feature_importance.head(15).to_string(
        index=False
    )
)


print("\n✅ DECISION TREE ANALYSIS COMPLETED!")

TRAINING DECISION TREE MODEL

✅ Decision Tree model trained successfully!

📌 DECISION TREE PERFORMANCE
------------------------------------------------------------
Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1-Score  : 1.0000
ROC-AUC   : 1.0000

📌 CLASSIFICATION REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

  Not Placed       1.00      1.00      1.00      3188
      Placed       1.00      1.00      1.00      1812

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000


📌 CONFUSION MATRIX
------------------------------------------------------------
[[3188    0]
 [   0 1812]]

📌 DECISION TREE FEATURE IMPORTANCE
------------------------------------------------------------
                              Feature  Importance
      numerical__Communication_Skills    0.296993
  numerical__Career_Dev

In [6]:
# ============================================================
# STEP 4A: CHECKING FOR TARGET LEAKAGE
# ============================================================

print("=" * 60)
print("CHECKING FOR TARGET LEAKAGE")
print("=" * 60)


# ------------------------------------------------------------
# Check target-related column names
# ------------------------------------------------------------

print("\n📌 FEATURES WITH TARGET-RELATED NAMES")
print("-" * 60)

target_keywords = [
    "placement",
    "target",
    "label",
    "status",
    "prediction"
]

possible_leakage_columns = [
    col for col in X_train.columns
    if any(
        keyword.lower() in col.lower()
        for keyword in target_keywords
    )
]

if possible_leakage_columns:
    print(possible_leakage_columns)
else:
    print("No target-related feature names found.")


# ------------------------------------------------------------
# Check whether feature values are suspiciously related
# to the target
# ------------------------------------------------------------

print("\n📌 TARGET CORRELATION CHECK")
print("-" * 60)

# Convert target temporarily to binary
y_train_binary = (
    y_train
    .map({
        "Not Placed": 0,
        "Placed": 1
    })
)

# Calculate correlations
numeric_train = X_train.select_dtypes(
    include=["number"]
)

target_correlations = (
    numeric_train
    .corrwith(y_train_binary)
    .abs()
    .sort_values(ascending=False)
)

print(
    target_correlations.to_string()
)


# ------------------------------------------------------------
# Check Decision Tree depth and leaves
# ------------------------------------------------------------

print("\n📌 DECISION TREE COMPLEXITY")
print("-" * 60)

print(
    f"Tree depth   : {decision_tree_model.get_depth()}"
)

print(
    f"Number leaves: {decision_tree_model.get_n_leaves()}"
)


# ------------------------------------------------------------
# Check feature importance
# ------------------------------------------------------------

print("\n📌 TOP DECISION TREE FEATURES")
print("-" * 60)

print(
    tree_feature_importance.head(15).to_string(
        index=False
    )
)


print("\n✅ TARGET LEAKAGE CHECK COMPLETED!")

CHECKING FOR TARGET LEAKAGE

📌 FEATURES WITH TARGET-RELATED NAMES
------------------------------------------------------------
No target-related feature names found.

📌 TARGET CORRELATION CHECK
------------------------------------------------------------
numerical__Career_Development_Index      0.511868
numerical__Projects                      0.500809
numerical__CGPA                          0.491517
numerical__Backlogs                      0.491494
numerical__Certifications                0.474846
numerical__Practical_Experience_Index    0.469579
numerical__Academic_Strength             0.466819
numerical__Coding_Skills                 0.439393
numerical__Aptitude_Test_Score           0.390105
numerical__Communication_Skills          0.325837
numerical__Professional_Skill_Index      0.249188

📌 DECISION TREE COMPLEXITY
------------------------------------------------------------
Tree depth   : 6
Number leaves: 12

📌 TOP DECISION TREE FEATURES
-----------------------------------------

In [7]:
# ============================================================
# STEP 4B: TRAINING VS TESTING PERFORMANCE CHECK
# ============================================================

print("=" * 60)
print("CHECKING TRAINING VS TESTING PERFORMANCE")
print("=" * 60)


# ------------------------------------------------------------
# Training predictions
# ------------------------------------------------------------

tree_train_predictions = decision_tree_model.predict(
    X_train
)

tree_test_predictions = decision_tree_model.predict(
    X_test
)


# ------------------------------------------------------------
# Training performance
# ------------------------------------------------------------

train_accuracy = accuracy_score(
    y_train,
    tree_train_predictions
)

test_accuracy = accuracy_score(
    y_test,
    tree_test_predictions
)


print("\n📌 ACCURACY COMPARISON")
print("-" * 60)

print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")


# ------------------------------------------------------------
# Difference
# ------------------------------------------------------------

accuracy_difference = (
    train_accuracy - test_accuracy
)

print(
    f"\nTrain-Test Difference: "
    f"{accuracy_difference:.4f}"
)


# ------------------------------------------------------------
# Check exact predictions
# ------------------------------------------------------------

train_errors = (
    y_train != tree_train_predictions
).sum()

test_errors = (
    y_test != tree_test_predictions
).sum()


print("\n📌 MISCLASSIFICATION COUNT")
print("-" * 60)

print(f"Training errors : {train_errors}")
print(f"Testing errors  : {test_errors}")


print("\n✅ TRAINING VS TESTING CHECK COMPLETED!")

CHECKING TRAINING VS TESTING PERFORMANCE

📌 ACCURACY COMPARISON
------------------------------------------------------------
Training Accuracy : 1.0000
Testing Accuracy  : 1.0000

Train-Test Difference: 0.0000

📌 MISCLASSIFICATION COUNT
------------------------------------------------------------
Training errors : 0
Testing errors  : 0

✅ TRAINING VS TESTING CHECK COMPLETED!


In [8]:
# ============================================================
# STEP 5: RANDOM FOREST - PLACEMENT READINESS MODEL
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("=" * 60)
print("TRAINING RANDOM FOREST MODEL")
print("=" * 60)


# ------------------------------------------------------------
# Create Random Forest
# ------------------------------------------------------------

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

random_forest_model.fit(
    X_train,
    y_train
)

print("\n✅ Random Forest model trained successfully!")


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

rf_predictions = random_forest_model.predict(
    X_test
)

rf_probabilities = random_forest_model.predict_proba(
    X_test
)


# ------------------------------------------------------------
# Probability of Placed
# ------------------------------------------------------------

placed_class_index = list(
    random_forest_model.classes_
).index("Placed")

rf_placed_probability = (
    rf_probabilities[:, placed_class_index]
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

rf_precision = precision_score(
    y_test,
    rf_predictions,
    pos_label="Placed",
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_predictions,
    pos_label="Placed",
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    rf_predictions,
    pos_label="Placed",
    zero_division=0
)

rf_roc_auc = roc_auc_score(
    y_test,
    rf_placed_probability
)


# ------------------------------------------------------------
# Display performance
# ------------------------------------------------------------

print("\n📌 RANDOM FOREST PERFORMANCE")
print("-" * 60)

print(f"Accuracy  : {rf_accuracy:.4f}")
print(f"Precision : {rf_precision:.4f}")
print(f"Recall    : {rf_recall:.4f}")
print(f"F1-Score  : {rf_f1:.4f}")
print(f"ROC-AUC   : {rf_roc_auc:.4f}")


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\n📌 CLASSIFICATION REPORT")
print("-" * 60)

print(
    classification_report(
        y_test,
        rf_predictions,
        target_names=[
            "Not Placed",
            "Placed"
        ],
        zero_division=0
    )
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

print("\n📌 CONFUSION MATRIX")
print("-" * 60)

rf_cm = confusion_matrix(
    y_test,
    rf_predictions,
    labels=[
        "Not Placed",
        "Placed"
    ]
)

print(rf_cm)


# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

rf_feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance":
        random_forest_model.feature_importances_
})

rf_feature_importance = (
    rf_feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n📌 RANDOM FOREST FEATURE IMPORTANCE")
print("-" * 60)

print(
    rf_feature_importance.head(15).to_string(
        index=False
    )
)


print("\n✅ RANDOM FOREST ANALYSIS COMPLETED!")

TRAINING RANDOM FOREST MODEL

✅ Random Forest model trained successfully!

📌 RANDOM FOREST PERFORMANCE
------------------------------------------------------------
Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1-Score  : 1.0000
ROC-AUC   : 1.0000

📌 CLASSIFICATION REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

  Not Placed       1.00      1.00      1.00      3188
      Placed       1.00      1.00      1.00      1812

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000


📌 CONFUSION MATRIX
------------------------------------------------------------
[[3188    0]
 [   0 1812]]

📌 RANDOM FOREST FEATURE IMPORTANCE
------------------------------------------------------------
                              Feature  Importance
      numerical__Communication_Skills    0.278184
                  numer

“Tree-based models achieved perfect performance on the held-out test set, suggesting that the synthetic dataset contains highly deterministic relationships between student attributes and placement status. Logistic Regression provided a more conservative benchmark with ROC-AUC of 0.936.”

In [9]:
# ============================================================
# CHECK AVAILABLE DATAFRAMES
# ============================================================

print("=" * 60)
print("AVAILABLE DATAFRAMES")
print("=" * 60)

dataframes = [
    (name, obj)
    for name, obj in list(globals().items())
    if isinstance(obj, pd.DataFrame)
]

for name, obj in dataframes:
    print(f"{name:<35} {obj.shape}")

AVAILABLE DATAFRAMES
X_train                             (45000, 11)
X_test                              (5000, 11)
tree_feature_importance             (11, 2)
numeric_train                       (45000, 11)
rf_feature_importance               (11, 2)


In [10]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders available here:")
print(os.listdir())

Current working directory:
c:\Users\HARSHADA\Desktop\Career-Aspiration-Skill-Gap\notebook

Files/folders available here:
['01_Data_Understanding.ipynb', '02_Data_Cleaning.ipynb', '03_EDA.ipynb', '04_Skill_Gap_Analysis.ipynb', '05_Feature_Engineering.ipynb', '06_ML_Model.ipynb', '07_Model_Explainability.ipynb', '08_Recommendation_System.ipynb', 'data']


In [11]:
import os

print("=" * 60)
print("CHECKING DATA FOLDER")
print("=" * 60)

for root, dirs, files in os.walk("data"):
    print(f"\n📁 {root}")

    for file in files:
        print(f"   └── {file}")

CHECKING DATA FOLDER

📁 data

📁 data\processed
   └── career_fit_results.pkl
   └── career_fit_summary.csv
   └── feature_importance.csv
   └── feature_importance_model.pkl
   └── preprocessor.pkl
   └── selected_features.csv
   └── X_test_selected.csv
   └── X_train_selected.csv
   └── y_test.csv
   └── y_train.csv


In [12]:
# ============================================================
# STEP 6A: LOAD ORIGINAL STUDENT DATA
# ============================================================

import pandas as pd

print("=" * 60)
print("LOADING ORIGINAL STUDENT DATA")
print("=" * 60)

train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

print("\n📌 TRAINING DATA")
print("-" * 60)
print("Shape:", train_df.shape)

print("\n📌 TESTING DATA")
print("-" * 60)
print("Shape:", test_df.shape)

print("\n📌 ORIGINAL FEATURES")
print("-" * 60)

print(train_df.columns.tolist())

print("\n📌 SAMPLE DATA")
print("-" * 60)

print(train_df.head())

print("\n✅ ORIGINAL STUDENT DATA LOADED SUCCESSFULLY!")

LOADING ORIGINAL STUDENT DATA

📌 TRAINING DATA
------------------------------------------------------------
Shape: (45000, 15)

📌 TESTING DATA
------------------------------------------------------------
Shape: (5000, 15)

📌 ORIGINAL FEATURES
------------------------------------------------------------
['Student_ID', 'Age', 'Gender', 'Degree', 'Branch', 'CGPA', 'Internships', 'Projects', 'Coding_Skills', 'Communication_Skills', 'Aptitude_Test_Score', 'Soft_Skills_Rating', 'Certifications', 'Backlogs', 'Placement_Status']

📌 SAMPLE DATA
------------------------------------------------------------
   Student_ID  Age  Gender  Degree Branch  CGPA  Internships  Projects  \
0        1048   22  Female  B.Tech    ECE  6.29            0         3   
1       37820   20  Female     BCA    ECE  6.05            1         4   
2       49668   22    Male     MCA     ME  7.22            1         4   
3       19467   22    Male     MCA     ME  7.78            2         4   
4       23094   20  Female 

In [13]:
# ============================================================
# STEP 6A: PREPARING STUDENT PROFILES FOR CAREER FIT
# ============================================================

print("=" * 60)
print("PREPARING STUDENT PROFILES FOR CAREER FIT ANALYSIS")
print("=" * 60)


# ------------------------------------------------------------
# Load original training data if train_df is unavailable
# ------------------------------------------------------------

if "train_df" not in globals():

    train_df = pd.read_csv(
        "data/raw/train.csv"
    )

    print("✅ Original training data loaded.")

else:

    print("✅ Existing training data found.")


# ------------------------------------------------------------
# Create normalized student skill profiles
# ------------------------------------------------------------

normalized_profiles = pd.DataFrame()

normalized_profiles["Student_ID"] = (
    train_df["Student_ID"]
)


# ------------------------------------------------------------
# Academic performance
# CGPA is already on approximately 0–10 scale
# ------------------------------------------------------------

normalized_profiles["Academic_Score"] = (
    train_df["CGPA"]
)


# ------------------------------------------------------------
# Internship experience
# Maximum expected internships = 3
# ------------------------------------------------------------

normalized_profiles["Internship_Score"] = (
    train_df["Internships"]
    .clip(0, 3)
    / 3
    * 10
)


# ------------------------------------------------------------
# Project experience
# Maximum expected projects = 6
# ------------------------------------------------------------

normalized_profiles["Project_Score"] = (
    train_df["Projects"]
    .clip(0, 6)
    / 6
    * 10
)


# ------------------------------------------------------------
# Coding skills
# ------------------------------------------------------------

normalized_profiles["Coding_Score"] = (
    train_df["Coding_Skills"]
    .clip(0, 10)
)


# ------------------------------------------------------------
# Communication
# ------------------------------------------------------------

normalized_profiles["Communication_Score"] = (
    train_df["Communication_Skills"]
    .clip(0, 10)
)


# ------------------------------------------------------------
# Aptitude
# Convert 0–100 → 0–10
# ------------------------------------------------------------

normalized_profiles["Aptitude_Score"] = (
    train_df["Aptitude_Test_Score"]
    .clip(0, 100)
    / 10
)


# ------------------------------------------------------------
# Certifications
# Maximum expected certifications = 3
# ------------------------------------------------------------

normalized_profiles["Certification_Score"] = (
    train_df["Certifications"]
    .clip(0, 3)
    / 3
    * 10
)


# ------------------------------------------------------------
# Academic stability
# Fewer backlogs = stronger score
# ------------------------------------------------------------

normalized_profiles["Academic_Stability_Score"] = (
    (3 - train_df["Backlogs"].clip(0, 3))
    / 3
    * 10
)


# ------------------------------------------------------------
# Soft skills
# ------------------------------------------------------------

normalized_profiles["Soft_Skills_Score"] = (
    train_df["Soft_Skills_Rating"]
    .clip(0, 10)
)


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("\n📌 NORMALIZED STUDENT PROFILE")
print("-" * 60)

print(
    normalized_profiles.head(10).to_string(
        index=False
    )
)


print("\n📌 PROFILE SHAPE")
print("-" * 60)

print(
    normalized_profiles.shape
)


print("\n📌 SKILL SCORE SUMMARY")
print("-" * 60)

print(
    normalized_profiles
    .drop(columns=["Student_ID"])
    .describe()
    .round(2)
)


print("\n✅ STUDENT PROFILES READY FOR CAREER FIT ENGINE!")

PREPARING STUDENT PROFILES FOR CAREER FIT ANALYSIS
✅ Existing training data found.

📌 NORMALIZED STUDENT PROFILE
------------------------------------------------------------
 Student_ID  Academic_Score  Internship_Score  Project_Score  Coding_Score  Communication_Score  Aptitude_Score  Certification_Score  Academic_Stability_Score  Soft_Skills_Score
       1048            6.29          0.000000       5.000000             4                    6             5.1             3.333333                  0.000000                  5
      37820            6.05          3.333333       6.666667             6                    8             5.9             6.666667                  6.666667                  8
      49668            7.22          3.333333       6.666667             6                    6             5.8             6.666667                  3.333333                  6
      19467            7.78          6.666667       6.666667             6                    6             9.0   

In [14]:
# ============================================================
# STEP 16A: EXPANDED CAREER REQUIREMENT FRAMEWORK
# ============================================================

print("=" * 70)
print("BUILDING EXPANDED CAREER REQUIREMENT FRAMEWORK")
print("=" * 70)

career_requirements = {

    # ========================================================
    # 1. TECHNOLOGY & DATA
    # ========================================================

    "Data Scientist": {
        "domain": "Technology & Data",
        "Python": 8,
        "SQL": 7,
        "Statistics": 8,
        "Machine_Learning": 8,
        "Data_Visualization": 7,
        "Problem_Solving": 8,
        "Communication": 6,
        "Git_GitHub": 6,
        "Projects": 7,
        "Internship": 6
    },

    "Data Analyst": {
        "domain": "Technology & Data",
        "Excel": 8,
        "SQL": 8,
        "Statistics": 7,
        "Data_Visualization": 8,
        "Business_Analysis": 8,
        "Problem_Solving": 7,
        "Communication": 7,
        "Projects": 6,
        "Internship": 5
    },

    "Machine Learning Engineer": {
        "domain": "Technology & Data",
        "Python": 9,
        "Machine_Learning": 9,
        "Statistics": 7,
        "Deep_Learning": 8,
        "Data_Structures": 8,
        "Problem_Solving": 9,
        "Git_GitHub": 7,
        "Projects": 8,
        "Internship": 6
    },

    "AI Engineer": {
        "domain": "Technology & Data",
        "Python": 9,
        "Machine_Learning": 8,
        "Deep_Learning": 9,
        "Mathematics": 7,
        "Programming": 9,
        "Problem_Solving": 9,
        "Projects": 8,
        "Internship": 6
    },

    "Data Engineer": {
        "domain": "Technology & Data",
        "Python": 8,
        "SQL": 9,
        "Database_Management": 9,
        "ETL": 8,
        "Cloud": 7,
        "Data_Pipelines": 8,
        "Problem_Solving": 8,
        "Git_GitHub": 7,
        "Projects": 7,
        "Internship": 6
    },

    "Software Developer": {
        "domain": "Technology & Data",
        "Programming": 9,
        "Data_Structures": 9,
        "Algorithms": 8,
        "Database_Management": 7,
        "Problem_Solving": 9,
        "Git_GitHub": 8,
        "Projects": 8,
        "Internship": 6,
        "Communication": 6
    },

    "Cybersecurity Analyst": {
        "domain": "Technology & Data",
        "Cybersecurity": 9,
        "Networking": 8,
        "Operating_Systems": 8,
        "Risk_Analysis": 8,
        "Problem_Solving": 8,
        "Security_Tools": 8,
        "Programming": 6,
        "Projects": 6,
        "Internship": 6
    },

    "Cloud Engineer": {
        "domain": "Technology & Data",
        "Cloud_Computing": 9,
        "Networking": 8,
        "Linux": 8,
        "Infrastructure": 8,
        "Automation": 7,
        "Programming": 7,
        "Problem_Solving": 8,
        "Projects": 7,
        "Internship": 6
    },

    "DevOps Engineer": {
        "domain": "Technology & Data",
        "DevOps": 9,
        "Cloud_Computing": 8,
        "Linux": 8,
        "CI_CD": 9,
        "Automation": 8,
        "Git_GitHub": 8,
        "Programming": 7,
        "Problem_Solving": 8,
        "Projects": 7
    },

    "QA Engineer": {
        "domain": "Technology & Data",
        "Software_Testing": 9,
        "Test_Automation": 8,
        "Programming": 7,
        "Problem_Solving": 8,
        "Attention_to_Detail": 9,
        "Quality_Assurance": 9,
        "Projects": 6,
        "Internship": 5
    },


    # ========================================================
    # 2. BUSINESS & MANAGEMENT
    # ========================================================

    "Business Analyst": {
        "domain": "Business & Management",
        "Business_Analysis": 9,
        "Excel": 8,
        "SQL": 7,
        "Data_Visualization": 8,
        "Problem_Solving": 8,
        "Communication": 9,
        "Stakeholder_Management": 8,
        "Projects": 6,
        "Internship": 6
    },

    "Project Manager": {
        "domain": "Business & Management",
        "Project_Management": 9,
        "Leadership": 9,
        "Communication": 9,
        "Problem_Solving": 8,
        "Planning": 9,
        "Stakeholder_Management": 8,
        "Teamwork": 8,
        "Internship": 7
    },

    "Product Manager": {
        "domain": "Business & Management",
        "Product_Management": 9,
        "Market_Research": 8,
        "Business_Analysis": 8,
        "Communication": 9,
        "Leadership": 8,
        "Problem_Solving": 9,
        "Product_Strategy": 9,
        "Stakeholder_Management": 8,
        "Projects": 7
    },

    "Management Consultant": {
        "domain": "Business & Management",
        "Problem_Solving": 9,
        "Business_Analysis": 9,
        "Communication": 9,
        "Critical_Thinking": 9,
        "Research": 8,
        "Presentation": 8,
        "Excel": 7,
        "Internship": 6
    },

    "Operations Analyst": {
        "domain": "Business & Management",
        "Operations_Analysis": 9,
        "Excel": 8,
        "Data_Analysis": 8,
        "Process_Improvement": 8,
        "Problem_Solving": 8,
        "Communication": 7,
        "Business_Analysis": 7,
        "Projects": 6
    },

    "HR Analyst": {
        "domain": "Business & Management",
        "HR_Analytics": 9,
        "Excel": 8,
        "Data_Analysis": 7,
        "Communication": 8,
        "Employee_Relations": 8,
        "Problem_Solving": 7,
        "Reporting": 7,
        "Projects": 5
    },

    "Operations Manager": {
        "domain": "Business & Management",
        "Operations_Management": 9,
        "Leadership": 9,
        "Planning": 9,
        "Problem_Solving": 8,
        "Communication": 8,
        "Process_Improvement": 8,
        "Teamwork": 8,
        "Internship": 7
    },


    # ========================================================
    # 3. FINANCE & ACCOUNTING
    # ========================================================

    "Financial Analyst": {
        "domain": "Finance & Accounting",
        "Financial_Analysis": 9,
        "Financial_Modeling": 8,
        "Excel": 9,
        "Accounting": 8,
        "Statistics": 7,
        "Risk_Analysis": 7,
        "Problem_Solving": 8,
        "Communication": 7,
        "Internship": 6
    },

    "Accountant": {
        "domain": "Finance & Accounting",
        "Accounting": 9,
        "Financial_Reporting": 9,
        "Taxation": 8,
        "Excel": 8,
        "Auditing": 7,
        "Attention_to_Detail": 9,
        "Communication": 6,
        "Internship": 5
    },

    "Investment Analyst": {
        "domain": "Finance & Accounting",
        "Financial_Analysis": 9,
        "Financial_Modeling": 9,
        "Investment_Research": 9,
        "Valuation": 8,
        "Excel": 9,
        "Risk_Analysis": 8,
        "Statistics": 7,
        "Communication": 7,
        "Internship": 7
    },

    "Credit Analyst": {
        "domain": "Finance & Accounting",
        "Credit_Analysis": 9,
        "Financial_Analysis": 8,
        "Risk_Analysis": 9,
        "Accounting": 7,
        "Financial_Reporting": 7,
        "Excel": 8,
        "Problem_Solving": 8,
        "Communication": 7
    },

    "Risk Analyst": {
        "domain": "Finance & Accounting",
        "Risk_Analysis": 9,
        "Financial_Analysis": 8,
        "Statistics": 8,
        "Risk_Modeling": 8,
        "Excel": 8,
        "Problem_Solving": 8,
        "Critical_Thinking": 8,
        "Communication": 7
    },

    "Tax Consultant": {
        "domain": "Finance & Accounting",
        "Taxation": 9,
        "Accounting": 9,
        "Financial_Reporting": 7,
        "Excel": 8,
        "Regulatory_Knowledge": 9,
        "Attention_to_Detail": 9,
        "Communication": 7
    },

    "Financial Planner": {
        "domain": "Finance & Accounting",
        "Financial_Planning": 9,
        "Investment_Analysis": 8,
        "Risk_Analysis": 8,
        "Financial_Products": 8,
        "Communication": 9,
        "Client_Management": 9,
        "Problem_Solving": 8
    },


    # ========================================================
    # 4. MARKETING & MEDIA
    # ========================================================

    "Digital Marketing Specialist": {
        "domain": "Marketing & Media",
        "Digital_Marketing": 9,
        "SEO": 8,
        "Social_Media": 8,
        "Content_Strategy": 8,
        "Marketing_Analytics": 8,
        "Communication": 8,
        "Creativity": 8,
        "Projects": 6,
        "Internship": 6
    },

    "Content Strategist": {
        "domain": "Marketing & Media",
        "Content_Strategy": 9,
        "Writing": 9,
        "Research": 8,
        "SEO": 7,
        "Communication": 9,
        "Creativity": 9,
        "Social_Media": 7,
        "Portfolio": 7
    },

    "Social Media Manager": {
        "domain": "Marketing & Media",
        "Social_Media": 9,
        "Content_Creation": 8,
        "Communication": 9,
        "Creativity": 9,
        "Digital_Marketing": 8,
        "Analytics": 7,
        "Brand_Management": 8,
        "Projects": 6
    },

    "SEO Specialist": {
        "domain": "Marketing & Media",
        "SEO": 9,
        "Keyword_Research": 9,
        "Content_Strategy": 8,
        "Analytics": 8,
        "Digital_Marketing": 8,
        "Research": 8,
        "Communication": 7,
        "Projects": 6
    },

    "Brand Manager": {
        "domain": "Marketing & Media",
        "Brand_Management": 9,
        "Marketing_Strategy": 9,
        "Market_Research": 8,
        "Communication": 9,
        "Creativity": 8,
        "Consumer_Insights": 8,
        "Presentation": 8,
        "Projects": 6
    },

    "Market Research Analyst": {
        "domain": "Marketing & Media",
        "Market_Research": 9,
        "Consumer_Insights": 9,
        "Statistics": 8,
        "Data_Analysis": 8,
        "Research": 9,
        "Communication": 8,
        "Problem_Solving": 8,
        "Reporting": 7
    },

    "Public Relations Specialist": {
        "domain": "Marketing & Media",
        "Public_Relations": 9,
        "Communication": 9,
        "Writing": 9,
        "Media_Relations": 8,
        "Presentation": 8,
        "Creativity": 8,
        "Research": 7
    },


    # ========================================================
    # 5. DESIGN & CREATIVE
    # ========================================================

    "UI/UX Designer": {
        "domain": "Design & Creative",
        "UI_UX": 9,
        "Figma": 9,
        "Design_Thinking": 9,
        "User_Research": 8,
        "Visual_Design": 8,
        "Creativity": 9,
        "Communication": 7,
        "Portfolio": 9,
        "Projects": 8
    },

    "Graphic Designer": {
        "domain": "Design & Creative",
        "Visual_Design": 9,
        "Typography": 8,
        "Color_Theory": 8,
        "Adobe_Tools": 8,
        "Creativity": 9,
        "Branding": 8,
        "Portfolio": 9,
        "Projects": 8
    },

    "Product Designer": {
        "domain": "Design & Creative",
        "Product_Design": 9,
        "UI_UX": 9,
        "Figma": 9,
        "Design_Thinking": 9,
        "User_Research": 8,
        "Visual_Design": 8,
        "Creativity": 8,
        "Portfolio": 9
    },

    "Motion Designer": {
        "domain": "Design & Creative",
        "Motion_Design": 9,
        "Animation": 9,
        "Visual_Design": 8,
        "Adobe_Tools": 8,
        "Creativity": 9,
        "Storytelling": 8,
        "Portfolio": 9,
        "Projects": 7
    },

    "Video Editor": {
        "domain": "Design & Creative",
        "Video_Editing": 9,
        "Storytelling": 8,
        "Visual_Design": 8,
        "Creativity": 9,
        "Audio_Editing": 7,
        "Content_Creation": 8,
        "Portfolio": 9
    },

    "Creative Director": {
        "domain": "Design & Creative",
        "Creative_Direction": 9,
        "Leadership": 9,
        "Branding": 8,
        "Visual_Design": 8,
        "Creativity": 10,
        "Communication": 9,
        "Storytelling": 8,
        "Portfolio": 8
    },


    # ========================================================
    # 6. EDUCATION
    # ========================================================

    "Teacher": {
        "domain": "Education",
        "Subject_Knowledge": 9,
        "Teaching": 9,
        "Communication": 9,
        "Presentation": 8,
        "Classroom_Management": 8,
        "Patience": 9,
        "Planning": 8
    },

    "Instructional Designer": {
        "domain": "Education",
        "Instructional_Design": 9,
        "Content_Development": 8,
        "Communication": 8,
        "Research": 8,
        "Technology": 7,
        "Creativity": 8,
        "Presentation": 8
    },

    "Academic Counselor": {
        "domain": "Education",
        "Counseling": 9,
        "Communication": 9,
        "Career_Guidance": 9,
        "Active_Listening": 9,
        "Problem_Solving": 8,
        "Empathy": 9,
        "Planning": 7
    },

    "Corporate Trainer": {
        "domain": "Education",
        "Training": 9,
        "Communication": 9,
        "Presentation": 9,
        "Leadership": 8,
        "Content_Development": 8,
        "Public_Speaking": 9,
        "Planning": 8
    },

    "Education Content Developer": {
        "domain": "Education",
        "Content_Development": 9,
        "Subject_Knowledge": 8,
        "Writing": 9,
        "Research": 8,
        "Creativity": 8,
        "Communication": 8,
        "Technology": 7
    },


    # ========================================================
    # 7. SCIENCE & RESEARCH
    # ========================================================

    "Research Scientist": {
        "domain": "Science & Research",
        "Research_Methodology": 9,
        "Statistics": 8,
        "Scientific_Writing": 9,
        "Experimentation": 9,
        "Critical_Thinking": 9,
        "Data_Analysis": 8,
        "Problem_Solving": 8,
        "Communication": 7
    },

    "Research Analyst": {
        "domain": "Science & Research",
        "Research": 9,
        "Data_Analysis": 8,
        "Statistics": 8,
        "Critical_Thinking": 9,
        "Report_Writing": 8,
        "Problem_Solving": 8,
        "Communication": 7
    },

    "Lab Analyst": {
        "domain": "Science & Research",
        "Laboratory_Techniques": 9,
        "Data_Analysis": 8,
        "Experimentation": 9,
        "Scientific_Method": 9,
        "Attention_to_Detail": 9,
        "Documentation": 8,
        "Problem_Solving": 7
    },

    "Clinical Research Associate": {
        "domain": "Science & Research",
        "Clinical_Research": 9,
        "Research_Methodology": 9,
        "Documentation": 9,
        "Data_Analysis": 8,
        "Regulatory_Knowledge": 8,
        "Attention_to_Detail": 9,
        "Communication": 7
    },

    "Environmental Analyst": {
        "domain": "Science & Research",
        "Environmental_Science": 9,
        "Data_Analysis": 8,
        "Research": 8,
        "Statistics": 7,
        "Environmental_Assessment": 9,
        "Report_Writing": 8,
        "Problem_Solving": 8
    },


    # ========================================================
    # 8. HEALTHCARE
    # ========================================================

    "Healthcare Analyst": {
        "domain": "Healthcare",
        "Healthcare_Analytics": 9,
        "Data_Analysis": 8,
        "Statistics": 8,
        "Healthcare_Systems": 8,
        "Problem_Solving": 8,
        "Communication": 7,
        "Reporting": 8
    },

    "Clinical Data Analyst": {
        "domain": "Healthcare",
        "Clinical_Data": 9,
        "Data_Analysis": 9,
        "Statistics": 8,
        "Research_Methodology": 8,
        "Data_Quality": 9,
        "Reporting": 8,
        "Attention_to_Detail": 9
    },

    "Healthcare Administrator": {
        "domain": "Healthcare",
        "Healthcare_Management": 9,
        "Operations_Management": 8,
        "Leadership": 8,
        "Communication": 9,
        "Planning": 8,
        "Problem_Solving": 8,
        "Reporting": 7
    },

    "Medical Research Assistant": {
        "domain": "Healthcare",
        "Medical_Research": 9,
        "Research_Methodology": 8,
        "Data_Collection": 8,
        "Documentation": 9,
        "Statistics": 7,
        "Scientific_Writing": 8,
        "Attention_to_Detail": 9
    },


    # ========================================================
    # 9. LEGAL & COMPLIANCE
    # ========================================================

    "Legal Analyst": {
        "domain": "Legal & Compliance",
        "Legal_Research": 9,
        "Legal_Writing": 9,
        "Critical_Thinking": 9,
        "Research": 8,
        "Attention_to_Detail": 9,
        "Communication": 8,
        "Problem_Solving": 8
    },

    "Compliance Analyst": {
        "domain": "Legal & Compliance",
        "Compliance": 9,
        "Regulatory_Knowledge": 9,
        "Risk_Analysis": 8,
        "Documentation": 9,
        "Attention_to_Detail": 9,
        "Research": 8,
        "Communication": 7
    },

    "Risk & Compliance Associate": {
        "domain": "Legal & Compliance",
        "Risk_Management": 9,
        "Compliance": 9,
        "Regulatory_Knowledge": 8,
        "Risk_Analysis": 9,
        "Documentation": 8,
        "Problem_Solving": 8,
        "Communication": 7
    },

    "Corporate Governance Analyst": {
        "domain": "Legal & Compliance",
        "Corporate_Governance": 9,
        "Compliance": 8,
        "Legal_Research": 8,
        "Risk_Management": 8,
        "Documentation": 9,
        "Communication": 8,
        "Attention_to_Detail": 9
    },


    # ========================================================
    # 10. ENVIRONMENT & SUSTAINABILITY
    # ========================================================

    "Sustainability Analyst": {
        "domain": "Environment & Sustainability",
        "Sustainability": 9,
        "Data_Analysis": 8,
        "Environmental_Assessment": 8,
        "Research": 8,
        "Reporting": 8,
        "Problem_Solving": 8,
        "Communication": 7
    },

    "Environmental Consultant": {
        "domain": "Environment & Sustainability",
        "Environmental_Science": 9,
        "Environmental_Assessment": 9,
        "Research": 8,
        "Project_Management": 7,
        "Report_Writing": 8,
        "Problem_Solving": 8,
        "Communication": 8
    },

    "ESG Analyst": {
        "domain": "Environment & Sustainability",
        "ESG": 9,
        "Sustainability": 9,
        "Data_Analysis": 8,
        "Risk_Analysis": 8,
        "Reporting": 9,
        "Research": 8,
        "Communication": 7
    },

    "Renewable Energy Analyst": {
        "domain": "Environment & Sustainability",
        "Renewable_Energy": 9,
        "Energy_Analysis": 9,
        "Data_Analysis": 8,
        "Statistics": 7,
        "Environmental_Science": 8,
        "Research": 8,
        "Problem_Solving": 8
    }
}


# ============================================================
# FRAMEWORK VALIDATION
# ============================================================

print("\n✅ Career requirement framework created successfully!")

print("\nTotal careers defined:", len(career_requirements))

print("\nCareer Domains:")
from collections import Counter

domain_counts = Counter(
    career["domain"]
    for career in career_requirements.values()
)

for domain, count in domain_counts.items():
    print(f"  {domain}: {count} careers")


# ------------------------------------------------------------
# CAREER SKILL COUNTS
# ------------------------------------------------------------

print("\nCareer Skill Coverage:")
for career, requirements in career_requirements.items():

    skill_count = len(requirements) - 1

    print(
        f"{career:<35} "
        f"{skill_count:>2} skills"
    )


# ------------------------------------------------------------
# VALIDATION CHECKS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FRAMEWORK VALIDATION")
print("=" * 70)

# Check duplicate career names
career_names = list(career_requirements.keys())

if len(career_names) == len(set(career_names)):
    print("✅ No duplicate career names")


# Check domains
if all(
    "domain" in requirements
    for requirements in career_requirements.values()
):
    print("✅ All careers have a domain")


# Check skill requirement values
invalid_requirements = []

for career, requirements in career_requirements.items():

    for skill, value in requirements.items():

        if skill == "domain":
            continue

        if not isinstance(value, (int, float)) or not 0 <= value <= 10:
            invalid_requirements.append(
                (career, skill, value)
            )

if len(invalid_requirements) == 0:
    print("✅ All skill requirements are valid (0–10)")
else:
    print("⚠️ Invalid skill requirements found:")
    print(invalid_requirements)


# ------------------------------------------------------------
# SAMPLE CAREER REQUIREMENTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE CAREER REQUIREMENTS")
print("=" * 70)

sample_careers = [
    "Data Scientist",
    "Financial Analyst",
    "UI/UX Designer",
    "Teacher",
    "Legal Analyst",
    "Sustainability Analyst"
]

for career in sample_careers:

    print(f"\n{career}")
    print("-" * len(career))

    for skill, requirement in career_requirements[career].items():

        if skill != "domain":
            print(
                f"{skill:<30} Required Level: {requirement}/10"
            )


print("\n" + "=" * 70)
print("EXPANDED CAREER REQUIREMENT FRAMEWORK READY!")
print("=" * 70)

BUILDING EXPANDED CAREER REQUIREMENT FRAMEWORK

✅ Career requirement framework created successfully!

Total careers defined: 59

Career Domains:
  Technology & Data: 10 careers
  Business & Management: 7 careers
  Finance & Accounting: 7 careers
  Marketing & Media: 7 careers
  Design & Creative: 6 careers
  Education: 5 careers
  Science & Research: 5 careers
  Healthcare: 4 careers
  Legal & Compliance: 4 careers
  Environment & Sustainability: 4 careers

Career Skill Coverage:
Data Scientist                      10 skills
Data Analyst                         9 skills
Machine Learning Engineer            9 skills
AI Engineer                          8 skills
Data Engineer                       10 skills
Software Developer                   9 skills
Cybersecurity Analyst                9 skills
Cloud Engineer                       9 skills
DevOps Engineer                      9 skills
QA Engineer                          8 skills
Business Analyst                     9 skills
Project M

In [15]:
# ======================================================================
# STEP 17: BUILDING CAREER FIT SCORING ENGINE
# ======================================================================

print("=" * 70)
print("BUILDING CAREER FIT SCORING ENGINE")
print("=" * 70)

import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# CHECK REQUIRED DATA
# ----------------------------------------------------------------------

if "career_requirements" not in globals():
    raise NameError(
        "career_requirements is not available. "
        "Please run the Expanded Career Requirement Framework cell first."
    )

if "normalized_profiles" not in globals():
    raise NameError(
        "normalized_profiles is not available. "
        "Please run the Student Profile Preparation cell first."
    )

print("\n✅ Required data found.")

# ----------------------------------------------------------------------
# STUDENT SKILL MAPPING
# ----------------------------------------------------------------------

student_skill_mapping = {

    "Academic_Score": "Academic_Score",
    "Internship_Score": "Internship",
    "Project_Score": "Projects",

    "Coding_Score": "Programming",
    "Communication_Score": "Communication",
    "Aptitude_Score": "Problem_Solving",

    "Certification_Score": "Certifications",
    "Academic_Stability_Score": "Academic_Stability",
    "Soft_Skills_Score": "Soft_Skills"
}

print("\n📌 STUDENT PROFILE SKILLS")
print("-" * 70)

profile_skill_columns = [
    col for col in normalized_profiles.columns
    if col != "Student_ID"
]

for col in profile_skill_columns:
    print("✓", col)

# ----------------------------------------------------------------------
# CAREER FIT CALCULATION FUNCTION
# ----------------------------------------------------------------------

def calculate_career_fit(student, career_requirements):

    career_results = []

    for career_name, requirements in career_requirements.items():

        domain = requirements.get("domain", "Unknown")

        total_score = 0
        total_weight = 0

        skill_details = []

        for required_skill, required_level in requirements.items():

            # Skip metadata
            if required_skill == "domain":
                continue

            # ----------------------------------------------------------
            # FIND MATCHING STUDENT SKILL
            # ----------------------------------------------------------

            student_skill = None

            # Direct profile mapping
            if required_skill in student.index:
                student_skill = student[required_skill]

            # Mapping from profile skill names
            else:

                for profile_skill, career_skill in student_skill_mapping.items():

                    if career_skill == required_skill and profile_skill in student.index:
                        student_skill = student[profile_skill]
                        break

            # ----------------------------------------------------------
            # IF STUDENT SKILL IS NOT AVAILABLE
            # ----------------------------------------------------------

            if student_skill is None:
                continue

            student_skill = float(student_skill)
            required_level = float(required_level)

            # ----------------------------------------------------------
            # CAP STUDENT SCORE AT 10
            # ----------------------------------------------------------

            student_skill = max(0, min(10, student_skill))

            # ----------------------------------------------------------
            # SKILL MATCH SCORE
            # ----------------------------------------------------------

            match_score = min(
                student_skill / required_level * 100,
                100
            )

            gap = max(
                required_level - student_skill,
                0
            )

            total_score += match_score
            total_weight += 1

            skill_details.append({
                "Skill": required_skill,
                "Student_Score": round(student_skill, 2),
                "Required_Score": round(required_level, 2),
                "Gap": round(gap, 2),
                "Match_Percentage": round(match_score, 2)
            })

        # --------------------------------------------------------------
        # FINAL CAREER FIT SCORE
        # --------------------------------------------------------------

        if total_weight > 0:
            fit_score = total_score / total_weight
        else:
            fit_score = 0

        # --------------------------------------------------------------
        # CAREER READINESS CATEGORY
        # --------------------------------------------------------------

        if fit_score >= 80:
            readiness = "Excellent Fit"

        elif fit_score >= 65:
            readiness = "Strong Fit"

        elif fit_score >= 50:
            readiness = "Moderate Fit"

        else:
            readiness = "Needs Development"

        career_results.append({
            "Career": career_name,
            "Domain": domain,
            "Career_Fit_Score": round(fit_score, 2),
            "Readiness": readiness,
            "Skills_Evaluated": total_weight,
            "Skill_Details": skill_details
        })

    return career_results


# ----------------------------------------------------------------------
# TEST ON FIRST STUDENT
# ----------------------------------------------------------------------

first_student = normalized_profiles.iloc[0]

first_student_results = calculate_career_fit(
    first_student,
    career_requirements
)

first_student_results_df = pd.DataFrame(first_student_results)

first_student_results_df = first_student_results_df.sort_values(
    by="Career_Fit_Score",
    ascending=False
).reset_index(drop=True)

# ----------------------------------------------------------------------
# DISPLAY TOP CAREERS
# ----------------------------------------------------------------------

print("\n📌 TOP CAREER FITS FOR SAMPLE STUDENT")
print("-" * 70)

print(
    first_student_results_df[
        [
            "Career",
            "Domain",
            "Career_Fit_Score",
            "Readiness",
            "Skills_Evaluated"
        ]
    ].head(10).to_string(index=False)
)

print("\n📌 CAREER FIT RANGE")
print("-" * 70)

print(
    "Highest Career Fit Score :",
    first_student_results_df["Career_Fit_Score"].max()
)

print(
    "Lowest Career Fit Score  :",
    first_student_results_df["Career_Fit_Score"].min()
)

print(
    "Careers Evaluated        :",
    len(first_student_results_df)
)

print("\n" + "=" * 70)
print("CAREER FIT SCORING ENGINE CREATED SUCCESSFULLY!")
print("=" * 70)

BUILDING CAREER FIT SCORING ENGINE

✅ Required data found.

📌 STUDENT PROFILE SKILLS
----------------------------------------------------------------------
✓ Academic_Score
✓ Internship_Score
✓ Project_Score
✓ Coding_Score
✓ Communication_Score
✓ Aptitude_Score
✓ Certification_Score
✓ Academic_Stability_Score
✓ Soft_Skills_Score

📌 TOP CAREER FITS FOR SAMPLE STUDENT
----------------------------------------------------------------------
                     Career                       Domain  Career_Fit_Score     Readiness  Skills_Evaluated
             Tax Consultant         Finance & Accounting             85.71 Excellent Fit                 1
                ESG Analyst Environment & Sustainability             85.71 Excellent Fit                 1
         Compliance Analyst           Legal & Compliance             85.71 Excellent Fit                 1
Clinical Research Associate           Science & Research             85.71 Excellent Fit                 1
             SEO Speciali

In [16]:
# ======================================================================
# STEP 18: BUILDING CAREER-SPECIFIC SKILL MAPPING ENGINE
# ======================================================================

print("=" * 70)
print("BUILDING CAREER-SPECIFIC SKILL MAPPING ENGINE")
print("=" * 70)


# ----------------------------------------------------------------------
# 1. MAP STUDENT PROFILE DIMENSIONS TO CAREER-SPECIFIC SKILLS
# ----------------------------------------------------------------------

skill_mapping = {

    # ==============================================================
    # TECHNOLOGY & DATA
    # ==============================================================

    "Python": [
        "Coding_Score"
    ],

    "Programming": [
        "Coding_Score"
    ],

    "Data_Structures": [
        "Coding_Score",
        "Problem_Solving_Score"
    ],

    "Algorithms": [
        "Coding_Score",
        "Problem_Solving_Score"
    ],

    "SQL": [
        "Coding_Score",
        "Aptitude_Score"
    ],

    "Statistics": [
        "Aptitude_Score",
        "Academic_Score"
    ],

    "Machine_Learning": [
        "Coding_Score",
        "Academic_Score",
        "Project_Score"
    ],

    "Deep_Learning": [
        "Coding_Score",
        "Academic_Score",
        "Project_Score"
    ],

    "Data_Visualization": [
        "Project_Score",
        "Communication_Score"
    ],

    "Problem_Solving": [
        "Aptitude_Score",
        "Academic_Score"
    ],

    "Git_GitHub": [
        "Project_Score",
        "Internship_Score"
    ],

    "Projects": [
        "Project_Score"
    ],

    "Internship": [
        "Internship_Score"
    ],

    "Excel": [
        "Academic_Score",
        "Project_Score"
    ],

    "Business_Analysis": [
        "Aptitude_Score",
        "Communication_Score",
        "Project_Score"
    ],

    "Database_Management": [
        "Coding_Score",
        "Project_Score"
    ],

    "ETL": [
        "Project_Score",
        "Coding_Score"
    ],

    "Cloud": [
        "Project_Score",
        "Internship_Score"
    ],

    "Data_Pipelines": [
        "Project_Score",
        "Coding_Score"
    ],


    # ==============================================================
    # BUSINESS & MANAGEMENT
    # ==============================================================

    "Stakeholder_Management": [
        "Communication_Score",
        "Internship_Score"
    ],

    "Project_Management": [
        "Project_Score",
        "Internship_Score",
        "Communication_Score"
    ],

    "Leadership": [
        "Soft_Skills_Score",
        "Communication_Score",
        "Internship_Score"
    ],

    "Planning": [
        "Academic_Score",
        "Project_Score",
        "Soft_Skills_Score"
    ],

    "Teamwork": [
        "Soft_Skills_Score",
        "Communication_Score"
    ],

    "Critical_Thinking": [
        "Aptitude_Score",
        "Academic_Score"
    ],

    "Research": [
        "Academic_Score",
        "Project_Score"
    ],

    "Presentation": [
        "Communication_Score",
        "Project_Score"
    ],


    # ==============================================================
    # FINANCE & ACCOUNTING
    # ==============================================================

    "Financial_Analysis": [
        "Academic_Score",
        "Aptitude_Score",
        "Project_Score"
    ],

    "Financial_Modeling": [
        "Academic_Score",
        "Project_Score",
        "Aptitude_Score"
    ],

    "Accounting": [
        "Academic_Score",
        "Certification_Score"
    ],

    "Financial_Reporting": [
        "Academic_Score",
        "Certification_Score",
        "Communication_Score"
    ],

    "Taxation": [
        "Academic_Score",
        "Certification_Score"
    ],

    "Auditing": [
        "Academic_Stability_Score",
        "Certification_Score",
        "Academic_Score"
    ],

    "Attention_to_Detail": [
        "Academic_Stability_Score",
        "Certification_Score"
    ],

    "Investment_Research": [
        "Research_Score",
        "Academic_Score",
        "Aptitude_Score"
    ],

    "Valuation": [
        "Financial_Analysis_Score",
        "Academic_Score",
        "Aptitude_Score"
    ],

    "Risk_Analysis": [
        "Aptitude_Score",
        "Academic_Score"
    ],


    # ==============================================================
    # MARKETING & MEDIA
    # ==============================================================

    "Digital_Marketing": [
        "Project_Score",
        "Communication_Score"
    ],

    "SEO": [
        "Project_Score",
        "Academic_Score"
    ],

    "Social_Media": [
        "Communication_Score",
        "Project_Score"
    ],

    "Content_Strategy": [
        "Communication_Score",
        "Project_Score",
        "Creativity_Score"
    ],

    "Marketing_Analytics": [
        "Aptitude_Score",
        "Project_Score",
        "Academic_Score"
    ],

    "Creativity": [
        "Project_Score",
        "Soft_Skills_Score"
    ],

    "Writing": [
        "Communication_Score",
        "Academic_Score"
    ],

    "Content_Creation": [
        "Communication_Score",
        "Project_Score"
    ],

    "Brand_Management": [
        "Communication_Score",
        "Project_Score"
    ],

    "Analytics": [
        "Aptitude_Score",
        "Project_Score"
    ],


    # ==============================================================
    # DESIGN & CREATIVE
    # ==============================================================

    "UI_UX": [
        "Project_Score",
        "Creativity_Score"
    ],

    "Figma": [
        "Project_Score"
    ],

    "Design_Thinking": [
        "Problem_Solving_Score",
        "Project_Score"
    ],

    "User_Research": [
        "Research_Score",
        "Project_Score"
    ],

    "Visual_Design": [
        "Project_Score",
        "Creativity_Score"
    ],

    "Typography": [
        "Creativity_Score",
        "Project_Score"
    ],

    "Color_Theory": [
        "Creativity_Score",
        "Project_Score"
    ],

    "Adobe_Tools": [
        "Project_Score"
    ],

    "Branding": [
        "Creativity_Score",
        "Project_Score"
    ],

    "Portfolio": [
        "Project_Score"
    ],


    # ==============================================================
    # EDUCATION
    # ==============================================================

    "Subject_Knowledge": [
        "Academic_Score"
    ],

    "Teaching": [
        "Academic_Score",
        "Communication_Score"
    ],

    "Classroom_Management": [
        "Communication_Score",
        "Soft_Skills_Score"
    ],

    "Patience": [
        "Soft_Skills_Score"
    ],

    "Instructional_Design": [
        "Project_Score",
        "Creativity_Score"
    ],

    "Content_Development": [
        "Project_Score",
        "Communication_Score"
    ],

    "Technology": [
        "Coding_Score",
        "Project_Score"
    ],


    # ==============================================================
    # SCIENCE & RESEARCH
    # ==============================================================

    "Research_Methodology": [
        "Academic_Score",
        "Project_Score"
    ],

    "Scientific_Writing": [
        "Academic_Score",
        "Communication_Score"
    ],

    "Experimentation": [
        "Project_Score",
        "Academic_Score"
    ],

    "Data_Analysis": [
        "Aptitude_Score",
        "Academic_Score",
        "Project_Score"
    ],

    "Report_Writing": [
        "Communication_Score",
        "Academic_Score"
    ],


    # ==============================================================
    # GENERAL PROFESSIONAL SKILLS
    # ==============================================================

    "Communication": [
        "Communication_Score"
    ],

    "Soft_Skills": [
        "Soft_Skills_Score"
    ]
}


# ----------------------------------------------------------------------
# 2. HANDLE PROFILE DIMENSIONS NOT DIRECTLY PRESENT IN ORIGINAL DATA
# ----------------------------------------------------------------------

# These are proxy dimensions derived from existing student information.
# They allow career-specific requirements to be evaluated without
# pretending that the dataset directly measured every professional skill.

def get_proxy_score(student, proxy_skill):

    if proxy_skill == "Problem_Solving_Score":
        return (
            student["Aptitude_Score"] * 0.6 +
            student["Academic_Score"] * 0.4
        )

    elif proxy_skill == "Creativity_Score":
        return (
            student["Project_Score"] * 0.6 +
            student["Soft_Skills_Score"] * 0.4
        )

    elif proxy_skill == "Research_Score":
        return (
            student["Academic_Score"] * 0.6 +
            student["Project_Score"] * 0.4
        )

    elif proxy_skill == "Financial_Analysis_Score":
        return (
            student["Academic_Score"] * 0.5 +
            student["Aptitude_Score"] * 0.5
        )

    return None


# ----------------------------------------------------------------------
# 3. GET SCORE FOR A CAREER SKILL
# ----------------------------------------------------------------------

def calculate_skill_score(student, career_skill):

    required_sources = skill_mapping.get(career_skill, [])

    scores = []

    for source in required_sources:

        if source in student.index:
            scores.append(float(student[source]))

        else:
            proxy_score = get_proxy_score(student, source)

            if proxy_score is not None:
                scores.append(float(proxy_score))

    if len(scores) == 0:
        return None

    return round(sum(scores) / len(scores), 2)


# ----------------------------------------------------------------------
# 4. CREATE CAREER-SPECIFIC STUDENT SKILL PROFILE
# ----------------------------------------------------------------------

def build_career_skill_profile(student, career_name):

    requirements = career_requirements[career_name]

    skill_scores = {}

    for skill in requirements:

        if skill == "domain":
            continue

        score = calculate_skill_score(
            student,
            skill
        )

        if score is not None:
            skill_scores[skill] = score

    return skill_scores


# ----------------------------------------------------------------------
# 5. TEST WITH FIRST STUDENT
# ----------------------------------------------------------------------

sample_student = normalized_profiles.iloc[0]

print("\n📌 SAMPLE STUDENT")
print("-" * 70)

print(
    "Student ID:",
    sample_student["Student_ID"]
)


# ----------------------------------------------------------------------
# 6. SHOW CAREER-SPECIFIC SKILL MAPPING
# ----------------------------------------------------------------------

test_careers = [
    "Data Scientist",
    "Financial Analyst",
    "UI/UX Designer",
    "Teacher"
]

for career in test_careers:

    print("\n" + "-" * 70)
    print(career)
    print("-" * 70)

    profile = build_career_skill_profile(
        sample_student,
        career
    )

    for skill, score in profile.items():

        print(
            f"{skill:<30} {score:.2f}/10"
        )


# ----------------------------------------------------------------------
# 7. VALIDATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CAREER-SPECIFIC SKILL MAPPING VALIDATION")
print("=" * 70)

print(
    "\nTotal career definitions:",
    len(career_requirements)
)

print(
    "Total skill mappings:",
    len(skill_mapping)
)

print("\nExample domain-specific mapping:")

for skill in [
    "Financial_Analysis",
    "Accounting",
    "UI_UX",
    "Figma",
    "Python",
    "Machine_Learning"
]:

    print(
        f"✓ {skill:<25} → {skill_mapping.get(skill)}"
    )

print("\n" + "=" * 70)
print("CAREER-SPECIFIC SKILL MAPPING ENGINE READY!")
print("=" * 70)

BUILDING CAREER-SPECIFIC SKILL MAPPING ENGINE

📌 SAMPLE STUDENT
----------------------------------------------------------------------
Student ID: 1048.0

----------------------------------------------------------------------
Data Scientist
----------------------------------------------------------------------
Python                         4.00/10
SQL                            4.55/10
Statistics                     5.70/10
Machine_Learning               5.10/10
Data_Visualization             5.50/10
Problem_Solving                5.70/10
Communication                  6.00/10
Git_GitHub                     2.50/10
Projects                       5.00/10
Internship                     0.00/10

----------------------------------------------------------------------
Financial Analyst
----------------------------------------------------------------------
Financial_Analysis             5.46/10
Financial_Modeling             5.46/10
Excel                          5.64/10
Accounting          

In [17]:
# ======================================================================
# STEP 19: BUILDING CAREER FIT AND SKILL GAP ENGINE
# ======================================================================

print("=" * 70)
print("BUILDING CAREER FIT AND SKILL GAP ENGINE")
print("=" * 70)


import pandas as pd
import numpy as np


# ----------------------------------------------------------------------
# 1. VALIDATE REQUIRED OBJECTS
# ----------------------------------------------------------------------

required_objects = [
    "normalized_profiles",
    "career_requirements",
    "skill_mapping"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:

    raise NameError(
        f"Missing required objects: {missing_objects}"
    )

print("\n✅ Required objects found.")


# ----------------------------------------------------------------------
# 2. CAREER FIT CALCULATION FUNCTION
# ----------------------------------------------------------------------

def calculate_career_fit(student, career_name):

    requirements = career_requirements[career_name]

    evaluated_skills = []
    skill_details = []

    for skill, required_level in requirements.items():

        # Skip career domain
        if skill == "domain":
            continue

        student_score = calculate_skill_score(
            student,
            skill
        )

        # Only evaluate skills for which we have evidence/proxy
        if student_score is None:
            continue

        student_score = min(
            max(float(student_score), 0),
            10
        )

        required_level = min(
            max(float(required_level), 0),
            10
        )

        gap = max(
            required_level - student_score,
            0
        )

        match_percentage = (
            student_score / required_level
        ) * 100 if required_level > 0 else 0

        match_percentage = min(
            match_percentage,
            100
        )

        evaluated_skills.append(match_percentage)

        skill_details.append({
            "Skill": skill,
            "Student_Score": round(student_score, 2),
            "Required_Score": round(required_level, 2),
            "Skill_Gap": round(gap, 2),
            "Match_Percentage": round(
                match_percentage,
                2
            )
        })

    # --------------------------------------------------------------
    # Require enough evaluated skills
    # --------------------------------------------------------------

    if len(evaluated_skills) == 0:

        return None

    career_fit_score = np.mean(
        evaluated_skills
    )

    return {
        "Career_Fit_Score": round(
            career_fit_score,
            2
        ),
        "Skills_Evaluated": len(
            evaluated_skills
        ),
        "Skill_Details": skill_details
    }


# ----------------------------------------------------------------------
# 3. READINESS CLASSIFICATION
# ----------------------------------------------------------------------

def classify_readiness(score):

    if score >= 85:
        return "Excellent Fit"

    elif score >= 70:
        return "Strong Fit"

    elif score >= 55:
        return "Moderate Fit"

    elif score >= 40:
        return "Developing"

    else:
        return "Low Fit"


# ----------------------------------------------------------------------
# 4. BUILD CAREER FIT RESULTS FOR ALL STUDENTS
# ----------------------------------------------------------------------

career_fit_results = []

print("\nCalculating career fit scores...")


for student_idx, student in normalized_profiles.iterrows():

    student_id = student["Student_ID"]

    for career_name, career_info in career_requirements.items():

        result = calculate_career_fit(
            student,
            career_name
        )

        if result is None:
            continue

        career_fit_results.append({

            "Student_ID": student_id,

            "Career": career_name,

            "Domain": career_info["domain"],

            "Career_Fit_Score":
                result["Career_Fit_Score"],

            "Readiness":
                classify_readiness(
                    result["Career_Fit_Score"]
                ),

            "Skills_Evaluated":
                result["Skills_Evaluated"],

            "Skill_Details":
                result["Skill_Details"]
        })


career_fit_df = pd.DataFrame(
    career_fit_results
)


# ----------------------------------------------------------------------
# 5. VALIDATE RESULTS
# ----------------------------------------------------------------------

print("\n📌 CAREER FIT RESULT")
print("-" * 70)

print(
    "Total student-career evaluations:",
    len(career_fit_df)
)

print(
    "Unique students:",
    career_fit_df["Student_ID"].nunique()
)

print(
    "Unique careers:",
    career_fit_df["Career"].nunique()
)


# ----------------------------------------------------------------------
# 6. SAMPLE STUDENT RESULTS
# ----------------------------------------------------------------------

sample_student_id = normalized_profiles.iloc[0]["Student_ID"]

sample_results = (
    career_fit_df[
        career_fit_df["Student_ID"] == sample_student_id
    ]
    .sort_values(
        "Career_Fit_Score",
        ascending=False
    )
)


print("\n📌 TOP CAREER FITS FOR SAMPLE STUDENT")
print("-" * 70)

print(
    sample_results[
        [
            "Career",
            "Domain",
            "Career_Fit_Score",
            "Readiness",
            "Skills_Evaluated"
        ]
    ].head(10).to_string(
        index=False
    )
)


# ----------------------------------------------------------------------
# 7. SKILL GAP EXTRACTION
# ----------------------------------------------------------------------

def extract_skill_gaps(
    student_id,
    career_name,
    minimum_gap=1.0
):

    row = career_fit_df[
        (career_fit_df["Student_ID"] == student_id)
        &
        (career_fit_df["Career"] == career_name)
    ]

    if row.empty:
        return pd.DataFrame()

    details = row.iloc[0]["Skill_Details"]

    gap_df = pd.DataFrame(details)

    if gap_df.empty:
        return gap_df

    gap_df = gap_df[
        gap_df["Skill_Gap"] >= minimum_gap
    ]

    gap_df = gap_df.sort_values(
        "Skill_Gap",
        ascending=False
    )

    return gap_df


# ----------------------------------------------------------------------
# 8. SHOW SAMPLE STUDENT'S TOP CAREER + SKILL GAPS
# ----------------------------------------------------------------------

top_career = sample_results.iloc[0]["Career"]

print("\n📌 BEST CAREER FOR SAMPLE STUDENT")
print("-" * 70)

print(
    "Career:",
    top_career
)

print(
    "Domain:",
    sample_results.iloc[0]["Domain"]
)

print(
    "Career Fit Score:",
    sample_results.iloc[0]["Career_Fit_Score"]
)

print(
    "Readiness:",
    sample_results.iloc[0]["Readiness"]
)


print("\n📌 TOP SKILL GAPS")
print("-" * 70)

sample_gap_df = extract_skill_gaps(
    sample_student_id,
    top_career
)

print(
    sample_gap_df.head(10).to_string(
        index=False
    )
)


# ----------------------------------------------------------------------
# 9. CAREER RANKING VALIDATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CAREER FIT ENGINE VALIDATION")
print("=" * 70)

print(
    "\nCareer Fit Score range:"
)

print(
    "Minimum:",
    career_fit_df["Career_Fit_Score"].min()
)

print(
    "Maximum:",
    career_fit_df["Career_Fit_Score"].max()
)

print(
    "Average:",
    round(
        career_fit_df["Career_Fit_Score"].mean(),
        2
    )
)


# ----------------------------------------------------------------------
# 10. CHECK FOR THE PREVIOUS 85.71% PROBLEM
# ----------------------------------------------------------------------

print("\n📌 SKILLS-EVALUATED DISTRIBUTION")
print("-" * 70)

print(
    career_fit_df["Skills_Evaluated"]
    .value_counts()
    .sort_index()
)


# ----------------------------------------------------------------------
# 11. SAVE CAREER FIT RESULTS
# ----------------------------------------------------------------------

career_fit_df.to_pickle(
    "data/processed/career_fit_results.pkl"
)

print(
    "\n✓ Saved:",
    "data/processed/career_fit_results.pkl"
)


# ----------------------------------------------------------------------
# 12. SAVE BASIC CAREER FIT TABLE
# ----------------------------------------------------------------------

career_fit_summary = career_fit_df[
    [
        "Student_ID",
        "Career",
        "Domain",
        "Career_Fit_Score",
        "Readiness",
        "Skills_Evaluated"
    ]
]

career_fit_summary.to_csv(
    "data/processed/career_fit_summary.csv",
    index=False
)

print(
    "✓ Saved:",
    "data/processed/career_fit_summary.csv"
)


print("\n" + "=" * 70)
print("CAREER FIT AND SKILL GAP ENGINE CREATED SUCCESSFULLY!")
print("=" * 70)

BUILDING CAREER FIT AND SKILL GAP ENGINE

✅ Required objects found.

Calculating career fit scores...

📌 CAREER FIT RESULT
----------------------------------------------------------------------
Total student-career evaluations: 2655000
Unique students: 45000
Unique careers: 59

📌 TOP CAREER FITS FOR SAMPLE STUDENT
----------------------------------------------------------------------
                     Career                       Domain  Career_Fit_Score  Readiness  Skills_Evaluated
                 HR Analyst        Business & Management             80.99 Strong Fit                 5
         Operations Analyst        Business & Management             75.96 Strong Fit                 6
         Healthcare Analyst                   Healthcare             74.12 Strong Fit                 4
                ESG Analyst Environment & Sustainability             73.93 Strong Fit                 4
     Sustainability Analyst Environment & Sustainability             73.93 Strong Fit        

In [18]:
# ============================================================
# CHECK REQUIRED OBJECTS FOR STEP 20
# ============================================================

print("=" * 70)
print("CHECKING STEP 20 REQUIRED OBJECTS")
print("=" * 70)

required_objects = [
    "normalized_profiles",
    "career_requirements",
    "career_fit_df",
    "calculate_skill_score"
]

for obj in required_objects:
    if obj in globals():
        print(f"✅ {obj} is available")
    else:
        print(f"❌ {obj} is MISSING")

print("\n" + "=" * 70)

CHECKING STEP 20 REQUIRED OBJECTS
✅ normalized_profiles is available
✅ career_requirements is available
✅ career_fit_df is available
✅ calculate_skill_score is available



In [19]:
# ======================================================================
# STEP 20: OPTIMIZED PERSONALIZED CAREER RECOMMENDATION ENGINE
# ======================================================================

print("=" * 70)
print("BUILDING OPTIMIZED PERSONALIZED CAREER RECOMMENDATION ENGINE")
print("=" * 70)

import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# 1. CHECK REQUIRED OBJECTS
# ----------------------------------------------------------------------

required_objects = [
    "normalized_profiles",
    "career_requirements",
    "career_fit_df",
    "calculate_skill_score"
]

missing = [
    x for x in required_objects
    if x not in globals()
]

if missing:
    raise NameError(
        f"Missing required objects: {missing}"
    )

print("\n✅ All required objects found.")


# ----------------------------------------------------------------------
# 2. PREPARE CAREER COVERAGE
# ----------------------------------------------------------------------

print("\n📌 PREPARING CAREER COVERAGE")

career_skill_counts = {
    career: len([
        skill for skill in requirements
        if skill != "domain"
    ])
    for career, requirements
    in career_requirements.items()
}

career_fit_work = career_fit_df.copy()

career_fit_work["Required_Skills"] = (
    career_fit_work["Career"]
    .map(career_skill_counts)
)

career_fit_work["Skill_Coverage"] = (
    career_fit_work["Skills_Evaluated"]
    / career_fit_work["Required_Skills"]
    * 100
)

# Minimum 50% coverage
career_fit_work = career_fit_work[
    career_fit_work["Skill_Coverage"] >= 50
].copy()

print(
    "Evaluations after coverage filter:",
    len(career_fit_work)
)


# ----------------------------------------------------------------------
# 3. SELECT TOP CAREERS PER STUDENT
# ----------------------------------------------------------------------

print("\n📌 SELECTING TOP CAREER MATCHES")

# We don't need to deeply process all 2.3M evaluations.
# Keep top 10 careers for each student.

career_fit_work = career_fit_work.sort_values(
    ["Student_ID", "Career_Fit_Score"],
    ascending=[True, False]
)

top_career_candidates = (
    career_fit_work
    .groupby("Student_ID", sort=False)
    .head(10)
    .copy()
)

print(
    "Student-career candidates:",
    len(top_career_candidates)
)


# ----------------------------------------------------------------------
# 4. ADJUST FIT SCORE
# ----------------------------------------------------------------------

def adjusted_score(row):

    score = row["Career_Fit_Score"]
    coverage = row["Skill_Coverage"]

    if coverage >= 75:
        return round(score, 2)

    elif coverage >= 50:
        return round(score * 0.90, 2)

    return round(score * 0.70, 2)


top_career_candidates["Adjusted_Fit_Score"] = (
    top_career_candidates.apply(
        adjusted_score,
        axis=1
    )
)


# ----------------------------------------------------------------------
# 5. READINESS
# ----------------------------------------------------------------------

def readiness_label(score):

    if score >= 85:
        return "Excellent Fit"

    elif score >= 70:
        return "Strong Fit"

    elif score >= 55:
        return "Moderate Fit"

    elif score >= 40:
        return "Developing"

    return "Low Fit"


top_career_candidates["Readiness"] = (
    top_career_candidates[
        "Adjusted_Fit_Score"
    ].apply(readiness_label)
)


# ----------------------------------------------------------------------
# 6. EXTRACT TOP SKILL GAPS
# ----------------------------------------------------------------------

print("\n📌 EXTRACTING TOP SKILL GAPS")
print("Processing only top career candidates...")


def extract_gap_information(skill_details):

    if skill_details is None:
        return "", "", 0, 0

    try:

        gap_df = pd.DataFrame(skill_details)

        if gap_df.empty:
            return "", "", 0, 0

        if "Skill_Gap" not in gap_df.columns:
            return "", "", 0, 0

        gap_df = gap_df[
            gap_df["Skill_Gap"] > 0
        ].copy()

        if gap_df.empty:
            return "", "", 0, 0

        gap_df = gap_df.sort_values(
            "Skill_Gap",
            ascending=False
        )

        top = gap_df.head(5)

        names = top["Skill"].tolist()

        values = top["Skill_Gap"].tolist()

        high = int(
            (gap_df["Skill_Gap"] >= 3).sum()
        )

        medium = int(
            (
                (gap_df["Skill_Gap"] >= 1.5)
                &
                (gap_df["Skill_Gap"] < 3)
            ).sum()
        )

        return (
            ", ".join(names),
            ", ".join(
                [str(round(x, 2)) for x in values]
            ),
            high,
            medium
        )

    except Exception:
        return "", "", 0, 0


gap_information = (
    top_career_candidates[
        "Skill_Details"
    ]
    .apply(extract_gap_information)
)

top_career_candidates[
    "Top_Skill_Gaps"
] = gap_information.apply(lambda x: x[0])

top_career_candidates[
    "Top_Gap_Values"
] = gap_information.apply(lambda x: x[1])

top_career_candidates[
    "High_Priority_Gaps"
] = gap_information.apply(lambda x: x[2])

top_career_candidates[
    "Medium_Priority_Gaps"
] = gap_information.apply(lambda x: x[3])


# ----------------------------------------------------------------------
# 7. PERSONALIZED RECOMMENDATION
# ----------------------------------------------------------------------

def recommendation_text(gaps):

    if not gaps:
        return (
            "Student currently meets the evaluated "
            "requirements for this career."
        )

    return (
        "Career is suitable, but improvement is "
        "recommended in: "
        + ", ".join(gaps.split(", ")[:3])
    )


top_career_candidates[
    "Recommendation"
] = (
    top_career_candidates[
        "Top_Skill_Gaps"
    ].apply(recommendation_text)
)


# ----------------------------------------------------------------------
# 8. FINAL DATAFRAME
# ----------------------------------------------------------------------

career_recommendations_df = (
    top_career_candidates[
        [
            "Student_ID",
            "Career",
            "Domain",
            "Career_Fit_Score",
            "Skill_Coverage",
            "Adjusted_Fit_Score",
            "Readiness",
            "Top_Skill_Gaps",
            "Top_Gap_Values",
            "High_Priority_Gaps",
            "Medium_Priority_Gaps",
            "Recommendation"
        ]
    ]
    .rename(
        columns={
            "Career_Fit_Score":
                "Raw_Fit_Score"
        }
    )
    .copy()
)


# ----------------------------------------------------------------------
# 9. FINAL TOP 5 PER STUDENT
# ----------------------------------------------------------------------

career_recommendations_df = (
    career_recommendations_df
    .sort_values(
        ["Student_ID", "Adjusted_Fit_Score"],
        ascending=[True, False]
    )
    .groupby(
        "Student_ID",
        sort=False
    )
    .head(5)
    .reset_index(drop=True)
)


# ----------------------------------------------------------------------
# 10. SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("📌 PERSONALIZED CAREER RECOMMENDATION RESULT")
print("=" * 70)

print(
    "Total recommendations:",
    len(career_recommendations_df)
)

print(
    "Unique students:",
    career_recommendations_df[
        "Student_ID"
    ].nunique()
)

print(
    "Unique careers:",
    career_recommendations_df[
        "Career"
    ].nunique()
)


# ----------------------------------------------------------------------
# 11. SAMPLE STUDENT
# ----------------------------------------------------------------------

sample_student_id = (
    normalized_profiles.iloc[0]["Student_ID"]
)

sample_result = (
    career_recommendations_df[
        career_recommendations_df[
            "Student_ID"
        ] == sample_student_id
    ]
)

print("\n📌 TOP 5 CAREERS FOR SAMPLE STUDENT")
print("-" * 70)

print(
    sample_result[
        [
            "Career",
            "Domain",
            "Adjusted_Fit_Score",
            "Skill_Coverage",
            "Readiness",
            "Top_Skill_Gaps"
        ]
    ].to_string(index=False)
)


# ----------------------------------------------------------------------
# 12. SAVE
# ----------------------------------------------------------------------

career_recommendations_df.to_pickle(
    "data/processed/career_recommendations.pkl"
)

career_recommendations_df.to_csv(
    "data/processed/career_recommendations.csv",
    index=False
)

print("\n📁 SAVED FILES")
print("-" * 70)
print("✓ career_recommendations.pkl")
print("✓ career_recommendations.csv")


print("\n" + "=" * 70)
print("✅ STEP 20 COMPLETED SUCCESSFULLY!")
print("=" * 70)

BUILDING OPTIMIZED PERSONALIZED CAREER RECOMMENDATION ENGINE

✅ All required objects found.

📌 PREPARING CAREER COVERAGE
Evaluations after coverage filter: 2385000

📌 SELECTING TOP CAREER MATCHES
Student-career candidates: 450000

📌 EXTRACTING TOP SKILL GAPS
Processing only top career candidates...

📌 PERSONALIZED CAREER RECOMMENDATION RESULT
Total recommendations: 225000
Unique students: 45000
Unique careers: 50

📌 TOP 5 CAREERS FOR SAMPLE STUDENT
----------------------------------------------------------------------
            Career                Domain  Adjusted_Fit_Score  Skill_Coverage  Readiness                                                          Top_Skill_Gaps
Operations Analyst Business & Management               75.96            75.0 Strong Fit Data_Analysis, Excel, Problem_Solving, Business_Analysis, Communication
        HR Analyst Business & Management               72.89            62.5 Strong Fit                    Excel, Communication, Data_Analysis, Problem_Solv

In [20]:
# ======================================================================
# STEP 21: TOP CAREER RANKING & RECOMMENDATION SUMMARY
# ======================================================================

print("=" * 70)
print("BUILDING TOP CAREER RANKING SYSTEM")
print("=" * 70)

import pandas as pd
import numpy as np


# ======================================================================
# 1. VALIDATE DATA
# ======================================================================

if "career_recommendations_df" not in globals():

    raise NameError(
        "career_recommendations_df is not available. "
        "Please run Step 20 first."
    )

print("\n✅ Career recommendation data found.")

print(
    "Students:",
    career_recommendations_df["Student_ID"].nunique()
)

print(
    "Careers:",
    career_recommendations_df["Career"].nunique()
)


# ======================================================================
# 2. RANK CAREERS FOR EACH STUDENT
# ======================================================================

print("\n📌 RANKING CAREERS FOR EACH STUDENT")
print("-" * 70)

career_ranking_df = (
    career_recommendations_df
    .sort_values(
        [
            "Student_ID",
            "Adjusted_Fit_Score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .copy()
)

career_ranking_df["Career_Rank"] = (
    career_ranking_df
    .groupby("Student_ID")
    .cumcount() + 1
)


# ======================================================================
# 3. KEEP TOP 5 CAREERS
# ======================================================================

top5_career_recommendations_df = (
    career_ranking_df[
        career_ranking_df["Career_Rank"] <= 5
    ]
    .copy()
)


print(
    "Total Top-5 recommendations:",
    len(top5_career_recommendations_df)
)

print(
    "Students covered:",
    top5_career_recommendations_df[
        "Student_ID"
    ].nunique()
)


# ======================================================================
# 4. SAMPLE STUDENT
# ======================================================================

sample_student_id = (
    top5_career_recommendations_df[
        "Student_ID"
    ].iloc[0]
)

sample_top5 = (
    top5_career_recommendations_df[
        top5_career_recommendations_df[
            "Student_ID"
        ] == sample_student_id
    ]
    .sort_values("Career_Rank")
)


print("\n📌 SAMPLE STUDENT TOP 5 CAREERS")
print("-" * 70)

print(
    sample_top5[
        [
            "Career_Rank",
            "Career",
            "Domain",
            "Adjusted_Fit_Score",
            "Skill_Coverage",
            "Readiness",
            "Top_Skill_Gaps"
        ]
    ].to_string(index=False)
)


# ======================================================================
# 5. BEST CAREER FOR EACH STUDENT
# ======================================================================

best_career_df = (
    top5_career_recommendations_df[
        top5_career_recommendations_df[
            "Career_Rank"
        ] == 1
    ]
    .copy()
)


print("\n📌 BEST CAREER SUMMARY")
print("-" * 70)

print(
    "Students with a recommended career:",
    best_career_df["Student_ID"].nunique()
)


# ======================================================================
# 6. MOST RECOMMENDED CAREERS
# ======================================================================

career_popularity = (
    best_career_df[
        "Career"
    ]
    .value_counts()
    .reset_index()
)

career_popularity.columns = [
    "Career",
    "Students_Recommended"
]


print("\n📌 MOST COMMON #1 CAREER RECOMMENDATIONS")
print("-" * 70)

print(
    career_popularity
    .head(10)
    .to_string(index=False)
)


# ======================================================================
# 7. DOMAIN DISTRIBUTION
# ======================================================================

domain_distribution = (
    best_career_df[
        "Domain"
    ]
    .value_counts()
    .reset_index()
)

domain_distribution.columns = [
    "Domain",
    "Students_Recommended"
]


print("\n📌 RECOMMENDED CAREER DOMAINS")
print("-" * 70)

print(
    domain_distribution
    .to_string(index=False)
)


# ======================================================================
# 8. SAVE OUTPUTS
# ======================================================================

top5_career_recommendations_df.to_csv(
    "data/processed/top5_career_recommendations.csv",
    index=False
)

top5_career_recommendations_df.to_pickle(
    "data/processed/top5_career_recommendations.pkl"
)

best_career_df.to_csv(
    "data/processed/best_career_by_student.csv",
    index=False
)

career_popularity.to_csv(
    "data/processed/career_recommendation_summary.csv",
    index=False
)

domain_distribution.to_csv(
    "data/processed/career_domain_summary.csv",
    index=False
)


# ======================================================================
# 9. FINAL OUTPUT
# ======================================================================

print("\n📌 OUTPUT FILES")
print("-" * 70)

print("✓ data/processed/top5_career_recommendations.csv")
print("✓ data/processed/top5_career_recommendations.pkl")
print("✓ data/processed/best_career_by_student.csv")
print("✓ data/processed/career_recommendation_summary.csv")
print("✓ data/processed/career_domain_summary.csv")


print("\n" + "=" * 70)
print("TOP CAREER RANKING SYSTEM CREATED SUCCESSFULLY!")
print("=" * 70)

BUILDING TOP CAREER RANKING SYSTEM

✅ Career recommendation data found.
Students: 45000
Careers: 50

📌 RANKING CAREERS FOR EACH STUDENT
----------------------------------------------------------------------
Total Top-5 recommendations: 225000
Students covered: 45000

📌 SAMPLE STUDENT TOP 5 CAREERS
----------------------------------------------------------------------
 Career_Rank               Career                Domain  Adjusted_Fit_Score  Skill_Coverage     Readiness                                                           Top_Skill_Gaps
           1   Operations Analyst Business & Management               85.73       75.000000 Excellent Fit       Excel, Data_Analysis, Problem_Solving, Projects, Business_Analysis
           2 Social Media Manager     Marketing & Media               83.02      100.000000    Strong Fit Creativity, Social_Media, Analytics, Content_Creation, Digital_Marketing
           3     Research Analyst    Science & Research               82.54      100.000000  

In [21]:
# ======================================================================
# STEP 22: SKILL GAP PRIORITIZATION & DEVELOPMENT PLAN
# ======================================================================

print("=" * 70)
print("BUILDING SKILL GAP PRIORITIZATION & DEVELOPMENT PLAN")
print("=" * 70)

import pandas as pd
import numpy as np


# ======================================================================
# 1. CHECK REQUIRED DATA
# ======================================================================

if "top5_career_recommendations_df" not in globals():

    raise NameError(
        "top5_career_recommendations_df is not available. "
        "Please run Step 21 first."
    )

print("\n✅ Top-5 career recommendation data found.")

print(
    "Students:",
    top5_career_recommendations_df[
        "Student_ID"
    ].nunique()
)


# ======================================================================
# 2. EXTRACT SKILL GAPS
# ======================================================================

print("\n📌 EXTRACTING SKILL GAPS")
print("-" * 70)


gap_rows = []

for _, row in top5_career_recommendations_df.iterrows():

    student_id = row["Student_ID"]
    career = row["Career"]
    domain = row["Domain"]
    fit_score = row["Adjusted_Fit_Score"]

    gaps = row["Top_Skill_Gaps"]

    gap_values = row["Top_Gap_Values"]

    if pd.isna(gaps) or str(gaps).strip() == "":
        continue

    gap_names = [
        x.strip()
        for x in str(gaps).split(",")
        if x.strip()
    ]

    if pd.isna(gap_values):
        values = []
    else:
        try:
            values = [
                float(x.strip())
                for x in str(gap_values).split(",")
                if x.strip()
            ]
        except:
            values = []

    for i, skill in enumerate(gap_names):

        gap_value = (
            values[i]
            if i < len(values)
            else np.nan
        )

        # --------------------------------------------------------------
        # Priority
        # --------------------------------------------------------------

        if pd.isna(gap_value):

            priority = "Unknown"

        elif gap_value >= 3:

            priority = "High"

        elif gap_value >= 1.5:

            priority = "Medium"

        else:

            priority = "Low"


        gap_rows.append({

            "Student_ID":
                student_id,

            "Career":
                career,

            "Domain":
                domain,

            "Career_Fit_Score":
                fit_score,

            "Skill":
                skill,

            "Skill_Gap":
                gap_value,

            "Priority":
                priority
        })


# ======================================================================
# 3. CREATE SKILL GAP DATAFRAME
# ======================================================================

skill_gap_df = pd.DataFrame(gap_rows)


print(
    "Total skill-gap records:",
    len(skill_gap_df)
)

print(
    "Unique students:",
    skill_gap_df[
        "Student_ID"
    ].nunique()
)

print(
    "Unique skills:",
    skill_gap_df[
        "Skill"
    ].nunique()
)


# ======================================================================
# 4. OVERALL SKILL GAP PRIORITY
# ======================================================================

print("\n📌 OVERALL SKILL GAP PRIORITY")
print("-" * 70)

priority_summary = (
    skill_gap_df[
        "Priority"
    ]
    .value_counts()
)

print(priority_summary)


# ======================================================================
# 5. MOST COMMON SKILL GAPS
# ======================================================================

print("\n📌 MOST COMMON SKILL GAPS")
print("-" * 70)

skill_gap_summary = (
    skill_gap_df
    .groupby("Skill")
    .agg(
        Students_Affected=("Student_ID", "nunique"),
        Average_Gap=("Skill_Gap", "mean"),
        Maximum_Gap=("Skill_Gap", "max")
    )
    .reset_index()
    .sort_values(
        [
            "Students_Affected",
            "Average_Gap"
        ],
        ascending=False
    )
)


print(
    skill_gap_summary
    .head(15)
    .to_string(index=False)
)


# ======================================================================
# 6. HIGH-PRIORITY SKILLS
# ======================================================================

print("\n📌 HIGH-PRIORITY SKILLS")
print("-" * 70)

high_priority_skills = (
    skill_gap_df[
        skill_gap_df["Priority"] == "High"
    ]
    .groupby("Skill")
    .agg(
        Students_Affected=("Student_ID", "nunique"),
        Average_Gap=("Skill_Gap", "mean")
    )
    .reset_index()
    .sort_values(
        "Students_Affected",
        ascending=False
    )
)


print(
    high_priority_skills
    .head(15)
    .to_string(index=False)
)


# ======================================================================
# 7. SAMPLE STUDENT DEVELOPMENT PLAN
# ======================================================================

sample_student_id = (
    top5_career_recommendations_df[
        "Student_ID"
    ].iloc[0]
)

sample_gaps = (
    skill_gap_df[
        skill_gap_df[
            "Student_ID"
        ] == sample_student_id
    ]
    .sort_values(
        "Skill_Gap",
        ascending=False
    )
)


print("\n📌 SAMPLE STUDENT SKILL DEVELOPMENT PLAN")
print("-" * 70)

print(
    sample_gaps[
        [
            "Career",
            "Skill",
            "Skill_Gap",
            "Priority"
        ]
    ]
    .head(15)
    .to_string(index=False)
)


# ======================================================================
# 8. CAREER-WISE SKILL GAP SUMMARY
# ======================================================================

career_skill_gap_summary = (
    skill_gap_df
    .groupby(
        [
            "Career",
            "Skill"
        ]
    )
    .agg(
        Students_Affected=("Student_ID", "nunique"),
        Average_Gap=("Skill_Gap", "mean")
    )
    .reset_index()
    .sort_values(
        "Average_Gap",
        ascending=False
    )
)


# ======================================================================
# 9. SAVE RESULTS
# ======================================================================

skill_gap_df.to_csv(
    "data/processed/skill_gap_analysis.csv",
    index=False
)

skill_gap_df.to_pickle(
    "data/processed/skill_gap_analysis.pkl"
)

skill_gap_summary.to_csv(
    "data/processed/skill_gap_summary.csv",
    index=False
)

high_priority_skills.to_csv(
    "data/processed/high_priority_skills.csv",
    index=False
)

career_skill_gap_summary.to_csv(
    "data/processed/career_skill_gap_summary.csv",
    index=False
)


# ======================================================================
# 10. FINAL OUTPUT
# ======================================================================

print("\n📁 OUTPUT FILES")
print("-" * 70)

print("✓ skill_gap_analysis.csv")
print("✓ skill_gap_analysis.pkl")
print("✓ skill_gap_summary.csv")
print("✓ high_priority_skills.csv")
print("✓ career_skill_gap_summary.csv")


print("\n" + "=" * 70)
print("✅ STEP 22 COMPLETED SUCCESSFULLY!")
print("=" * 70)

BUILDING SKILL GAP PRIORITIZATION & DEVELOPMENT PLAN

✅ Top-5 career recommendation data found.
Students: 45000

📌 EXTRACTING SKILL GAPS
----------------------------------------------------------------------
Total skill-gap records: 890673
Unique students: 44372
Unique skills: 70

📌 OVERALL SKILL GAP PRIORITY
----------------------------------------------------------------------
Priority
Low       437355
Medium    348056
High      105262
Name: count, dtype: int64

📌 MOST COMMON SKILL GAPS
----------------------------------------------------------------------
              Skill  Students_Affected  Average_Gap  Maximum_Gap
    Problem_Solving              37572     1.309457         5.00
              Excel              37501     1.746850         5.92
      Communication              36983     2.139197         7.00
      Data_Analysis              36092     1.445187         4.88
      Risk_Analysis              33494     1.883562         5.00
  Business_Analysis              30217     1.

In [23]:
# ======================================================================
# FIX: LOAD TOP CAREER RANKING DATA FOR STEP 23
# ======================================================================

import os
import pandas as pd

print("=" * 70)
print("CHECKING TOP CAREER RANKING DATA")
print("=" * 70)

# Check common variable names
possible_names = [
    "top_career_df",
    "top_careers_df",
    "career_ranking_df",
    "top_career_recommendations_df",
    "top5_career_df"
]

found = None

for name in possible_names:
    if name in globals():
        found = name
        break

if found:
    top_career_df = globals()[found]
    print(f"✅ Found existing data: {found}")
    print("Shape:", top_career_df.shape)

else:
    # Try loading the saved Step 21 output
    possible_files = [
        "data/processed/top_career_ranking.csv",
        "data/processed/top_careers.csv",
        "data/processed/top5_career_recommendations.csv",
        "data/processed/career_ranking.csv"
    ]

    for file in possible_files:
        if os.path.exists(file):
            top_career_df = pd.read_csv(file)
            print(f"✅ Loaded: {file}")
            print("Shape:", top_career_df.shape)
            found = file
            break

if found is None:
    raise FileNotFoundError(
        "Top career ranking data could not be found. "
        "Run the Step 21 ranking cell once."
    )

print("\n📌 COLUMNS")
print(top_career_df.columns.tolist())

print("\n" + "=" * 70)
print("✅ TOP CAREER DATA READY FOR STEP 23")
print("=" * 70)

CHECKING TOP CAREER RANKING DATA
✅ Found existing data: career_ranking_df
Shape: (225000, 13)

📌 COLUMNS
['Student_ID', 'Career', 'Domain', 'Raw_Fit_Score', 'Skill_Coverage', 'Adjusted_Fit_Score', 'Readiness', 'Top_Skill_Gaps', 'Top_Gap_Values', 'High_Priority_Gaps', 'Medium_Priority_Gaps', 'Recommendation', 'Career_Rank']

✅ TOP CAREER DATA READY FOR STEP 23


In [24]:
# ======================================================================
# STEP 23 PRE-CHECK
# ======================================================================

print("=" * 70)
print("CHECKING STEP 23 REQUIRED DATA")
print("=" * 70)

possible_skill_gap_objects = [
    "skill_gap_df",
    "skill_gaps_df",
    "skill_gap_priority_df",
    "skill_development_df",
    "top_skill_gaps_df"
]

found_objects = []

for name in possible_skill_gap_objects:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame):
            found_objects.append((name, obj.shape))

print("\n📌 AVAILABLE SKILL-GAP DATA")
print("-" * 70)

if found_objects:
    for name, shape in found_objects:
        print(f"✅ {name}: {shape}")
else:
    print("❌ No skill-gap DataFrame found.")

print("\n📌 CAREER RANKING DATA")
print("-" * 70)
print("✅ career_ranking_df:", career_ranking_df.shape)

print("\n" + "=" * 70)

CHECKING STEP 23 REQUIRED DATA

📌 AVAILABLE SKILL-GAP DATA
----------------------------------------------------------------------
✅ skill_gap_df: (890673, 7)

📌 CAREER RANKING DATA
----------------------------------------------------------------------
✅ career_ranking_df: (225000, 13)

